# Multi-Agent Collaboration and Agent Protocols

Session 1 argued that most problems do **not** need multiple agents. This session covers the minority that do, and the plumbing that makes a team useful beyond a demo.

Both labs tell one story.

**Lab A** builds a five-agent research team around a supervisor, with a reviewer and a fact-checker that can send work back.

**Lab B** exposes that team's knowledge source — and the project folder around it — over the **Model Context Protocol**, so *any* client (a LangGraph agent, Claude Desktop, a colleague's IDE) can use it without a new integration each time. Then you plug the MCP tool back into the team.

A short **appendix** extends the same map to two more protocols: **A2A**, for delegating to an agent, and **AP2**, for cryptographically authorising and evidencing an agent-performed purchase inside a commerce flow.

|| Part | You build | Key idea |
|------|------|-----------|----------|
| **A** |**A1 · Team state & permissions** | A typed state plus enforced write-permissions per agent | An agent boundary is a *permission*, not a paragraph in a prompt |
| |**A2 · Five specialists** | Planner, Researcher, Writer, Fact-Checker, Reviewer — all model-backed | Scope the context, not just the prompt |
| |**A3 · The supervisor** | An LLM router wrapped around a pure, testable policy | The model may *route*; it may not *authorise* |
| |**A4 · Loop-backs & caps** | Revision loop with a hard revision ceiling | A loop-back edge without a counter is an outage |
| |**A5 · Why the critics exist** | A deliberately hallucinating writer, caught | The one thing five agents bought you |
| **B** |**B1 · A FastMCP server** | Sandboxed project-folder tools + one resource | Your docstring and type hints *are* the API contract |
| |**B2 · MultiServerMCPClient** | Discovery and invocation from LangChain | M×N integrations become M+N |
| |**B3 · An agent over MCP** | `create_agent` picking MCP tools itself | The model chooses the tool; the sandbox chooses the limits |
| |**B4 · Stateful sessions** | `client.session(...)` and raw `ClientSession` | When one long-lived connection beats many short ones |
| |**B5 · Bring it together** | The team's Researcher swaps to the MCP tool | **Milestone 6** |
| **Appendix** |**A2A & AP2** | A mocked cross-vendor purchase: quote an agent you didn't build, pay for it under a budget | MCP is agent→tool; A2A is agent→agent; AP2 secures agent-performed payments. This scenario composes A2A and AP2; neither requires the other |

### Every agent here is a real model call

There are no fake agents in this notebook. Planner, Researcher, Writer, Fact-Checker, Reviewer **and** the Supervisor all call a live model through **`ChatLiteLLM`**.

If your API key does not work, please contact the instructors. There are fallbacks, so each node falls back to a hand-written implementation, prints that it did so, and the graph keeps running. Every cell executes and every `assert` passes either way. Missing key costs you the *experience* of a live agent, but not the lab.

> **The principle to carry to work:** *let the model produce; let deterministic code decide.* You will see it three times: the Planner's output is filtered against a whitelist, the Fact-Checker's verdict is overruled by a regex, and the Supervisor's route is rejected if it is not legal in the current state.

Every cell ends with an `EXPECTED OUTPUT` block. Where a cell is model-driven the wording will vary run to run; the block shows a **representative** run and the notes tell you what must hold regardless.

### Steps to Obtain API Keys

**1. Google (Gemini AI Studio) API Key**

  * Go to [Google AI Studio](https://aistudio.google.com/).
  * Sign in with your Google account.
  * On the left-hand navigation menu, click on **API keys**.
  * Click the **Create API key** button present at the top right corner.
  * Select an existing Google Cloud project or create a new one, then generate and copy your `GOOGLE_API_KEY`.

## Setup

In [ ]:
# Pinned for this cohort - do not un-pin, so a framework update can't break the lab mid-session.
# langgraph              : the graph runtime (state, nodes, edges, conditional routing)
# langchain              : create_agent - the prebuilt tool-calling (ReAct) agent used in Lab B
# langchain-litellm      : ChatLiteLLM - LiteLLM wrapped as a LangChain chat model, so it speaks
#                          messages and supports .bind_tools() / .with_structured_output()
# fastmcp                : the ergonomic MCP SERVER framework (decorators -> protocol)
# langchain-mcp-adapters : the CLIENT side - turns MCP tools into LangChain/LangGraph tools
# mcp                    : the low-level protocol SDK (we look at it once, in B4)
# Version skew between an MCP server and its client is the #1 setup snag in this lab:
# the handshake fails with a message about protocol versions, not about your code.
# (No -q: if an install fails you want the full pip error visible, not silenced.)
%pip install "langgraph>=1.2,<2.0" "langchain>=1.3,<2.0" "langchain-core>=1.0" "python-dotenv>=1.0" "cryptography>=46,<47"
%pip install "litellm>=1.93,<2.0" "langchain-litellm>=0.7,<0.8"
%pip install "fastmcp>=3.4,<4.0" "mcp>=1.28" "langchain-mcp-adapters>=0.3,<0.4"
%pip install "mermaid-py"

print("Dependencies installed. If pip asks you to restart the kernel, do so, then continue.")

"""
EXPECTED OUTPUT
---------------
(pip install log)
Dependencies installed. If pip asks you to restart the kernel, do so, then continue.
"""

In [ ]:
# Readiness check - confirm every moving part imports BEFORE you build on it.
from dotenv import load_dotenv
load_dotenv(".env", override=True)  # reads OPENAI_API_KEY etc. from a .env copied from .env.template
import litellm

litellm._turn_on_debug()

import sys, os
checks = {}

def probe(label, fn):
    """Import in a try/except so ONE missing package doesn't hide the status of the others."""
    try:
        fn()
        checks[label] = "ok"
    except Exception as e:
        checks[label] = f"FAILED: {type(e).__name__}: {e}"

probe("langgraph core", lambda: __import__("langgraph.graph", fromlist=["StateGraph"]).StateGraph)
probe("langchain create_agent", lambda: __import__("langchain.agents", fromlist=["create_agent"]).create_agent)
probe("ChatLiteLLM", lambda: __import__("langchain_litellm", fromlist=["ChatLiteLLM"]).ChatLiteLLM)
probe("fastmcp (server)", lambda: __import__("fastmcp", fromlist=["FastMCP"]).FastMCP)
probe("mcp SDK (client)", lambda: __import__("mcp", fromlist=["ClientSession"]).ClientSession)
probe("langchain-mcp-adapters", lambda: __import__("langchain_mcp_adapters.client", fromlist=["MultiServerMCPClient"]).MultiServerMCPClient)
probe("cryptography", lambda: __import__("cryptography"))

print("Environment readiness")
print("---------------------")
for k, v in checks.items():
    print(f"  {k:>22} : {v}")

print(f"\n  python interpreter   : {sys.executable}")
print("  ^ Lab B launches the MCP server as a CHILD PROCESS using THIS interpreter. If you launch")
print("    it with a bare 'python', a different environment starts, the imports fail, and the")
print("    handshake dies with an unhelpful error. Always use sys.executable.")

if all(v == "ok" for v in checks.values()):
    print("\nCore is green. Run the next cell to connect a model, then start Lab A.")
else:
    print("\nFix the FAILED lines before continuing.")

"""
EXPECTED OUTPUT
---------------
Environment readiness
---------------------
         langgraph core : ok
  langchain create_agent : ok
            ChatLiteLLM : ok
        fastmcp (server) : ok
        mcp SDK (client) : ok
  langchain-mcp-adapters : ok

  python interpreter   : .../.venv/Scripts/python.exe
  ^ Lab B launches the MCP server as a CHILD PROCESS using THIS interpreter. ...

Core is green. Run the next cell to connect a model, then start Lab A.
"""

### Connecting the model — one handle used by the whole notebook

`ChatLiteLLM` is [LiteLLM](https://docs.litellm.ai/) — this program's standard LLM client — presented as a LangChain **chat model**. That matters for two concrete reasons:

1. **LangGraph nodes want a chat model.** Nodes are happiest talking to something that speaks message objects and supports `.bind_tools()` — which is exactly what a chat model gives you, and exactly what `create_agent` requires in Lab B.
2. **Provider portability.** Change one string (`LLM_MODEL`) and the same code runs against OpenAI, Anthropic, Gemini, Bedrock or a local Ollama model. That portability is the entire reason we route through LiteLLM rather than a provider SDK.

The cell below builds that handle and then **actually calls the model once**. A key that is present but *invalid* is the most common Day-1 failure, and a flag that only checks `os.getenv(...)` would happily lie to you about it.

Two helpers wrap every model call in this notebook:

* **`ask(system, user)`** — plain text in, plain text out. Returns `None` on any failure.
* **`ask_structured(system, user, Schema)`** — returns a **validated Pydantic object**, via `chat_model.with_structured_output(Schema)`. Under the hood LangChain converts your Pydantic class into a tool schema, forces the model to "call" it, and parses the arguments back into your class. This is how you stop parsing prose with regexes: when a node's output has to *drive routing*, it must be a typed object, not a paragraph you hope starts with the word "yes".

Both return `None` rather than raising, so every caller can be written as:

```python
value = ask_structured(...) or <deterministic fallback>
```

That one line is what keeps this notebook runnable without a key.

In [ ]:
GEMINI_API_KEY = "paste-your-free-key-here"  # Paste your free key here

In [ ]:
# The single model handle for this notebook. EVERY model-backed node below uses ask()/ask_structured().
from langchain_litellm import ChatLiteLLM
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Literal
import os
import time

# Swap the string to switch provider:
LLM_MODEL = "gemini/gemini-3.1-flash-lite"  # Change model here

# temperature=0 => as close to reproducible as a real model gets. This notebook is meant to be
# re-run and compared, so we never want creative variation in the control path.
chat_model = ChatLiteLLM(model=LLM_MODEL, temperature=0, api_key=os.getenv("GEMINI_API_KEY", GEMINI_API_KEY))

# Small pause after every real model call, so a chatty run (the 5-agent star makes several calls
# per turn) doesn't trip the provider's requests-per-minute limit. Tune to taste.
LLM_CALL_PAUSE_SECONDS = 3.0

def _probe_model() -> bool:
    """One real, tiny call. A present-but-invalid key fails HERE, once, with a clear message -
    instead of failing inside a graph node twenty cells later where it looks like a LangGraph bug."""
    try:
        chat_model.invoke([HumanMessage(content="Reply with the single word: ok")])
        return True
    except Exception as e:
        print(f"  model unavailable ({type(e).__name__}: {str(e)[:110]})")
        return False

LLM_ENABLED = _probe_model()

def ask(system: str, user: str) -> str | None:
    """Plain text -> text. Returns None (never raises) so callers can fall back deterministically."""
    if not LLM_ENABLED:
        return None
    try:
        msgs = [SystemMessage(content=system), HumanMessage(content=user)]
        return chat_model.invoke(msgs).content.strip()
    except Exception as e:                 # transient 429 / timeout: degrade, do not crash the lab
        print(f"  [model call failed: {type(e).__name__}] falling back")
        return None
    finally:
        time.sleep(LLM_CALL_PAUSE_SECONDS)  # give the rate limit room to breathe before the next call

# with_structured_output() builds a NEW runnable per schema. Building it is cheap but not free,
# so we cache one per schema class - a habit worth keeping in any real agent.
_STRUCTURED_CACHE: dict[type, object] = {}

def ask_structured(system: str, user: str, schema: type[BaseModel]) -> BaseModel | None:
    """Text -> a VALIDATED Pydantic object. Returns None on any failure.

    Why this instead of ask() + json.loads()? Because the model's answer here has to drive
    control flow (which topic, approved or not, which agent next). Pydantic validation is the
    boundary where "the model said something plausible" becomes "the program has a value it
    can branch on". If validation fails, you get None and take the deterministic path -
    you never route on a half-parsed string."""
    if not LLM_ENABLED:
        return None
    try:
        runnable = _STRUCTURED_CACHE.setdefault(schema, chat_model.with_structured_output(schema))
        return runnable.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    except Exception as e:
        print(f"  [structured call failed: {type(e).__name__}] falling back")
        return None
    finally:
        time.sleep(LLM_CALL_PAUSE_SECONDS)  # give the rate limit room to breathe before the next call

print("LLM_ENABLED :", LLM_ENABLED, "|", LLM_MODEL if LLM_ENABLED else "running on deterministic fallbacks")
print("Fallback mode is fully supported: every cell runs and every self-check passes either way.")

"""
EXPECTED OUTPUT
---------------
LLM_ENABLED : True | openai/gpt-oss-20b
Fallback mode is fully supported: every cell runs and every self-check passes either way.

(without a valid key you instead see:
   model unavailable (AuthenticationError: ...)
 LLM_ENABLED : False | running on deterministic fallbacks
 ... which is a supported way to run this notebook.)
"""


In [ ]:
# The program-standard graph visualiser. House rule: the moment you compile a graph, LOOK at it.
# In a five-agent star topology a mis-wired conditional edge is invisible in the code and
# obvious in the picture.
from mermaid import Mermaid
from IPython.display import display

def show_graph(app, title: str = ""):
    if title:
        print(title)
        print("-" * len(title))
    display(Mermaid(app.get_graph().draw_mermaid()))

print("show_graph() ready.")

"""
EXPECTED OUTPUT
---------------
show_graph() ready.
"""

---
# Lab A · A research team with a supervisor

## Some team patterns, and when each is right

| Pattern | Shape | Choose it when | Cost you accept |
|---|---|---|---|
| **Supervisor** | One coordinator routes work to specialists; specialists never talk to each other | You need traceability, and the routing rule is expressible | The supervisor is a bottleneck and every result passes through its context |
| **Choreography** (network / swarm) | Agents hand off peer-to-peer, no central authority (Like KIMI agent swarms) | Handoffs are genuinely dynamic and no one agent can know the routing rule | Control flow becomes emergent — hard to bound, hard to debug |
| **Actor–critic** (reviewer) | A producer creates, a separate critic evaluates, work loops back | Quality is checkable and the check should not share the producer's blind spots | Latency and cost of the extra pass; risk of infinite revision |

You will build a **supervisor** with **two embedded critics** (Fact-Checker and Reviewer) — the combination most common in production.

> **Analogy.** Supervisor = a team lead assigning tickets: everything routes through one desk, so there is one place to look when output is wrong. Choreography = an open-plan office where anyone taps anyone on the shoulder — fast, but nobody can reconstruct who decided what. Actor–critic = a sub-editor who never writes the article, and is useful *because* they didn't: a critic sharing the writer's context inherits the writer's blind spots.

### Why five agents, not one

Apply what we learnt from session 1. The trigger that holds is **seperation of expertise**: a fact-checker sharing the writer's context will confirm the writer's invented citation. The weaker triggers: the planner and writer share most of their context and could defensibly be one agent.

In [ ]:
# The knowledge base, written to CSV. It is a real file on disk for a reason: in Lab B this SAME
# file is served over MCP, so the swap from a local function to a protocol tool changes the
# transport and nothing else.
import csv

ROWS = [
 ("S1","vendor_concentration","Payments vendor register",
  "Ninety-one percent of card transactions route through a single processor, Northgate Pay."),
 ("S2","vendor_concentration","Contract exposure summary",
  "The Northgate Pay master agreement auto-renews annually with a ninety-day termination notice."),
 ("S3","incident_history","Incident post-mortem 2025-11",
  "A four-hour Northgate Pay outage in November 2025 halted eighty-eight percent of checkout volume."),
 ("S4","mitigation","Failover design note",
  "A secondary processor can be integrated behind the existing payment abstraction in about six weeks."),
 ("S5","mitigation","Cost model",
  "Maintaining a warm secondary processor adds an estimated forty thousand per year in fixed fees."),
 ("S6","contract_terms","Procurement standard",
  "Any vendor above sixty percent of transaction volume requires an approved concentration waiver."),
]

with open("knowledge_base.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["id", "topic", "title", "text"])
    w.writerows(ROWS)

with open("knowledge_base.csv", newline="", encoding="utf-8") as fh:
    KB = list(csv.DictReader(fh))

def search_kb(topic: str) -> list[dict]:
    """Local retrieval. In Lab B this exact behaviour moves behind an MCP tool."""
    return [r for r in KB if r["topic"] == topic]

BRIEF = ("Assess our vendor concentration risk in the payments stack: how exposed are we, "
         "what has gone wrong before, and what could we do about it?")

print(f"{len(KB)} records, topics: {sorted({r['topic'] for r in KB})}")
print("search_kb('incident_history') ->", [r["id"] for r in search_kb("incident_history")])

"""
EXPECTED OUTPUT
---------------
6 records, topics: ['contract_terms', 'incident_history', 'mitigation', 'vendor_concentration']
search_kb('incident_history') -> ['S3']
"""

---
## A1 · Team state, and agent boundaries as *permissions*

In Session 1 you wrote a pattern decision record with a **boundary** field: `what does each agent know that the others must not need?`

Two mechanisms, both required:

1. **Read scoping** — each agent receives only the state slice it needs. This keeps contexts small and tool selection accurate, and it is the real reason multi-agent helps.
2. **Write scoping** — each agent may update only its own keys, enforced by a decorator that raises rather than corrupting state silently.

Reuse Session 1's state rules: **accumulating fields** get a reducer; **control fields** (the ones the supervisor routes on) must not, or they never empty and the graph loops forever.

One permission deserves argument: the **Writer may reset `fact_check` and `review`**. This is deliberate — when the writer changes the artefact, every prior approval of it is void.

In [ ]:
# Team state, agent scopes, and the enforcement decorator.
from typing import TypedDict, Annotated
from operator import add
import inspect, re, json

MAX_REVISIONS = 2          # the loop budget. Everything about convergence hangs off this number.

class TeamState(TypedDict):
    brief: str
    plan: list[str]                       # control: which topics to research
    findings: Annotated[list[dict], add]  # AUDIT: accumulates across researcher calls
    draft: str
    fact_check: dict                      # control: {} = not yet checked
    review: dict                          # control: {} = not yet reviewed
    revision_count: int                   # the loop guard
    next_agent: str                       # the supervisor's decision, written into state
    status: str
    log: Annotated[list[str], add]        # AUDIT: the team's trajectory

# NOTE which fields have a reducer and which do not. `findings` and `log` use Annotated[..., add]
# so every node's return is APPENDED. `fact_check` / `review` deliberately have NO reducer, because
# the writer must be able to reset them to {} - a control field you cannot empty is a loop you
# cannot exit. That single distinction causes most "my LangGraph never terminates" bugs.

# The boundary, as data. Every agent may also append to "log" - the audit trail is shared.
AGENT_SCOPES = {
    "planner":      {"plan"},
    "researcher":   {"findings"},
    "writer":       {"draft", "revision_count", "fact_check", "review"},  # may VOID approvals
    "fact_checker": {"fact_check"},
    "reviewer":     {"review"},
    "supervisor":   {"next_agent"},
    "escalate":     {"status"},
}

def scoped(role: str):
    """Enforce write permissions. A violation raises instead of silently corrupting state.

    Handles async nodes too - in Lab B the researcher becomes async (it awaits an MCP call), and
    a decorator that only understood sync functions would wrap a coroutine and fail obscurely."""
    allowed = AGENT_SCOPES[role] | {"log"}
    def enforce(update):
        illegal = set(update) - allowed
        if illegal:
            raise PermissionError(
                f"agent '{role}' wrote outside its scope: {sorted(illegal)}; allowed={sorted(allowed)}")
        return update
    def decorate(fn):
        if inspect.iscoroutinefunction(fn):
            async def awrapper(state):
                return enforce(await fn(state))
            awrapper.__name__ = fn.__name__
            return awrapper
        def wrapper(state):
            return enforce(fn(state))
        wrapper.__name__ = fn.__name__
        return wrapper
    return decorate

for role, keys in AGENT_SCOPES.items():
    print(f"  {role:<13} may write: {sorted(keys)}")

"""
EXPECTED OUTPUT
---------------
  planner       may write: ['plan']
  researcher    may write: ['findings']
  writer        may write: ['draft', 'fact_check', 'review', 'revision_count']
  fact_checker  may write: ['fact_check']
  reviewer      may write: ['review']
  supervisor    may write: ['next_agent']
  escalate      may write: ['status']
"""

In [ ]:
# Self-check A1 - the boundary must actually bite.
@scoped("reviewer")
def rogue_reviewer(state):
    # A reviewer that "helpfully" fixes the draft itself. Extremely common failure: the
    # critic starts producing, and now nothing independent is checking the output.
    return {"review": {"approved": True}, "draft": "I rewrote it myself"}

try:
    rogue_reviewer({})
    raise AssertionError("the scope decorator did not fire")
except PermissionError as e:
    print("PermissionError as expected:")
    print(" ", e)

print("\nA critic that can edit the artefact is no longer a critic. The decorator makes that")
print("design claim enforceable instead of aspirational.")

"""
EXPECTED OUTPUT
---------------
PermissionError as expected:
  agent 'reviewer' wrote outside its scope: ['draft']; allowed=['log', 'review']

A critic that can edit the artefact is no longer a critic. The decorator makes that
design claim enforceable instead of aspirational.
"""

---
## A2 · Five specialists

Each agent below follows the **same shape**:

```python
@scoped("planner")                                   # 1. WRITE permission, enforced
def planner_node(state: TeamState) -> dict:
    ctx = context_for("planner", state)              # 2. READ scope: only this agent's slice
    out = ask_structured(SYSTEM_PROMPTS["planner"],  # 3. the model produces (None if unavailable)
                         f"Brief:\n{ctx['brief']}", Plan)
    plan = _validate(out) or _planner_fallback(ctx)  # 4. deterministic code decides
    return {"plan": plan, "log": [...]}              #    a PARTIAL update - never the whole state
```

Four things worth internalising from that shape:

* **A node returns a partial update, not the state.** LangGraph merges it, applying each field's reducer. Returning the whole state is the fastest way to clobber another agent's work.
* **The read slice is the real boundary.** The Fact-Checker's slice contains *no* brief and *no* plan — it can only compare citations against retrieved findings. That narrowness is the point: a fact-checker that knows what the document is *trying* to say will rationalise a wrong citation.
* **Step 4 is not optional.** Every model output in this notebook passes through validation before it is allowed to affect state: the Planner's topics are filtered against a whitelist, the Fact-Checker's verdict is overruled by a regex, the Supervisor's route is discarded if illegal.
* **The system prompts below are real.** They are what you actually tune, review and version at work — read them critically, especially for what each agent is told *not* to do.

In [ ]:
# The scoped system prompts and the READ-scoping function.
SYSTEM_PROMPTS = {
 "planner": "You break a research brief into 2-4 retrieval topics drawn ONLY from the "
            "available topic list. You do not research, write, or judge quality.",
 "researcher": "You choose exactly ONE topic to retrieve next, from a list of topics that have "
               "not been retrieved yet. You never summarise, interpret, or write prose.",
 "writer": "You compose a short executive briefing strictly from the supplied findings. Every "
           "factual sentence must carry a citation tag like [S3] referring to a supplied record. "
           "You never introduce facts that are not in the findings.",
 "fact_checker": "You receive a draft and a list of source records. You verify that every "
                 "citation tag in the draft refers to a supplied record. You do not judge "
                 "style, completeness, or usefulness.",
 "reviewer": "You judge STRUCTURE and completeness against the plan: is every planned topic "
             "covered, is there a Risks section, is it within length? You do not verify facts.",
 "supervisor": "You are the coordinator of a research team. Given the current state of the work, "
               "you choose which single specialist should act next. You never do the work yourself.",
}

def context_for(role: str, state: TeamState) -> dict:
    """READ scoping: hand each agent only the slice it needs.

    This is where multi-agent actually pays - small, sharp contexts - and it is invisible if you
    only scope prompts. Note what is ABSENT from each slice; that is the design.
    Also note this is plain Python. There is no framework magic here: 'context engineering' is
    mostly just deciding, per agent, which dictionary keys it gets to see."""
    if role == "planner":      return {"brief": state["brief"]}
    if role == "researcher":   return {"plan": state["plan"],
                                       "already": [f["topic"] for f in state["findings"]]}
    if role == "writer":       return {"brief": state["brief"], "plan": state["plan"],
                                       "findings": state["findings"], "problems": state["fact_check"]}
    # the fact-checker gets NO brief and NO plan: it cannot be persuaded by intent
    if role == "fact_checker": return {"draft": state["draft"], "findings": state["findings"]}
    if role == "reviewer":     return {"draft": state["draft"], "plan": state["plan"]}
    return {}

demo = {"brief": BRIEF, "plan": ["vendor_concentration"], "findings": KB[:2],
        "draft": "x", "fact_check": {}, "review": {}}
for role in ("planner", "fact_checker", "writer"):
    print(f"  {role:<13} sees {sorted(context_for(role, demo))}")
print("\nThe fact-checker cannot see the brief. It therefore cannot be persuaded by intent.")

"""
EXPECTED OUTPUT
---------------
  planner       sees ['brief']
  fact_checker  sees ['draft', 'findings']
  writer        sees ['brief', 'findings', 'plan', 'problems']

The fact-checker cannot see the brief. It therefore cannot be persuaded by intent.
"""

In [ ]:
# The typed contracts each agent must return, plus the deterministic floor under each one.

# Why Pydantic classes and not "just ask for JSON"? Because these values steer control flow.
# with_structured_output() turns each class into a tool schema, forces the model to fill it in,
# and validates the result - so a node either gets a well-formed object or gets None. There is no
# third case where you route on half-parsed prose. The Field(description=...) strings are NOT
# comments: they are sent to the model as part of the schema and are your best lever on quality.

class Plan(BaseModel):
    """The planner's output."""
    topics: list[str] = Field(description="2-4 retrieval topics, chosen ONLY from the allowed list")

class NextTopic(BaseModel):
    """The researcher's choice of what to retrieve this turn."""
    topic: str = Field(description="exactly one topic from the not-yet-retrieved list")

class FactCheck(BaseModel):
    """The fact-checker's verdict."""
    unsupported: list[str] = Field(description="citation tags like S9 that are NOT in the sources")
    reasoning: str = Field(description="one short sentence explaining the verdict")

class Review(BaseModel):
    """The reviewer's structural judgement."""
    approved: bool = Field(description="true only if every required section is present")
    notes: list[str] = Field(description="specific, actionable structural problems; empty if approved")

class Route(BaseModel):
    """The supervisor's routing decision."""
    next_agent: Literal["planner", "researcher", "writer", "fact_checker",
                        "reviewer", "escalate", "done"] = Field(description="who acts next")
    why: str = Field(description="one short clause justifying the choice")

# ----- the deterministic floor -------------------------------------------------------------
# Each of these is what runs when there is no key, or a call fails. They are also the *reference
# behaviour* - when a live agent misbehaves, diff it against these.

TOPIC_KEYWORDS = {                       # insertion order = deterministic plan order
    "vendor_concentration": ["concentration", "exposed", "dependency"],
    "incident_history":     ["gone wrong", "outage", "incident"],
    "mitigation":           ["do about it", "mitigat", "options"],
    "contract_terms":       ["contract", "waiver"],
}
ALLOWED_TOPICS = list(TOPIC_KEYWORDS)    # the whitelist the planner is filtered against

def _planner_fallback(brief: str) -> list[str]:
    b = brief.lower()
    return [t for t, kws in TOPIC_KEYWORDS.items() if any(k in b for k in kws)]

def _compose_fallback(state: TeamState, drop_tags=()) -> str:
    """A template writer. Deterministic, so the loop-back is reproducible without a model."""
    out = [f"# Briefing: {state['brief'][:46]}...", ""]
    for topic in state["plan"]:
        out.append(f"## {topic.replace('_', ' ').title()}")
        for f in [x for x in state["findings"] if x["topic"] == topic]:
            out.append(f"{f['text']} [{f['id']}]")
        out.append("")
    out.append("## Risks")
    out.append("Concentration on a single processor is the dominant risk. [S1]")
    # A plausible, well-written, entirely unsupported sentence. This is exactly what a real
    # writer-agent produces, and exactly what a fact-checker exists to catch. Leaving it in the
    # fallback means the no-key run still tells the whole story.
    out.append("Northgate Pay is the regional market leader and unlikely to fail. [S9]")
    text = "\n".join(out)
    if drop_tags:
        text = "\n".join(l for l in text.split("\n") if not any(f"[{t}]" in l for t in drop_tags))
    return text

def _structure_notes(draft: str, plan: list[str]) -> list[str]:
    """The reviewer's objective half: which required sections are missing, and is it too long."""
    notes = [f"missing section: {t}" for t in plan
             if f"## {t.replace('_', ' ').title()}" not in draft]
    if "## Risks" not in draft:
        notes.append("missing Risks section")
    if len(draft.split()) > 320:
        notes.append(f"too long: {len(draft.split())} words")
    return notes

def required_headers(plan: list[str]) -> str:
    """The exact section headers the writer must emit - shared by writer and reviewer so the
    producer and the critic cannot disagree about the spec itself, only about compliance."""
    return "\n".join(f"## {t.replace('_', ' ').title()}" for t in plan) + "\n## Risks"

print("schemas:", [c.__name__ for c in (Plan, NextTopic, FactCheck, Review, Route)])
print("allowed topics:", ALLOWED_TOPICS)
print("fallback plan for the brief:", _planner_fallback(BRIEF))

"""
EXPECTED OUTPUT
---------------
schemas: ['Plan', 'NextTopic', 'FactCheck', 'Review', 'Route']
allowed topics: ['vendor_concentration', 'incident_history', 'mitigation', 'contract_terms']
fallback plan for the brief: ['vendor_concentration', 'incident_history', 'mitigation']
"""

In [ ]:
# The PLANNER - decompose a brief into retrieval topics.
@scoped("planner")
def planner_node(state: TeamState) -> dict:
    ctx = context_for("planner", state)          # sees only the brief

    out = ask_structured(
        SYSTEM_PROMPTS["planner"] + f" The ONLY allowed topics are: {ALLOWED_TOPICS}.",
        f"Brief:\n{ctx['brief']}\n\nWhich of the allowed topics must be retrieved to answer it?",
        Plan)

    # DECIDE deterministically. Two guards, and both matter in production:
    #   1. intersect with the whitelist  -> a hallucinated topic ("competitor_analysis") that the
    #      knowledge base cannot answer would send the researcher into an empty retrieval forever;
    #   2. re-order to ALLOWED_TOPICS order -> downstream sections come out in a stable order, so
    #      two runs of the same brief are comparable.
    plan = []
    if out:
        picked = set(out.topics)
        plan = [t for t in ALLOWED_TOPICS if t in picked]

    source = "llm"
    if not plan:                                  # no key, failed call, or nothing survived the filter
        plan, source = _planner_fallback(ctx["brief"]), "fallback"

    return {"plan": plan,
            "log": [f"planner[{source}]: {len(plan)} step(s) -> {plan}"]}

print(planner_node({"brief": BRIEF, "findings": []})["log"][0])

"""
EXPECTED OUTPUT  (representative - the model may legitimately pick a different subset)
---------------
planner[llm]: 3 step(s) -> ['vendor_concentration', 'incident_history', 'mitigation']

What must hold either way: every topic is in ALLOWED_TOPICS, and the order follows it.
Without a key you see planner[fallback] with the same three topics.
"""

In [ ]:
# The RESEARCHER - retrieve exactly ONE unresearched topic per turn.
#
# Why one topic per turn, when a loop inside the node would be faster? Because the supervisor's
# routing becomes OBSERVABLE: you see "supervisor -> researcher" three times in the trace and can
# point at the decision that produced each retrieval. A researcher that loops internally hides its
# work from the trace, and you debug it by reading prompts instead of reading a graph.
@scoped("researcher")
def researcher_node(state: TeamState) -> dict:
    ctx = context_for("researcher", state)       # sees the plan and what is already retrieved
    todo = [t for t in ctx["plan"] if t not in ctx["already"]]

    # The model chooses which topic to do next - a genuine (if small) act of agency.
    out = ask_structured(
        SYSTEM_PROMPTS["researcher"],
        f"Topics not yet retrieved: {todo}\nChoose exactly one to retrieve now.",
        NextTopic)

    # GUARD, and this one is load-bearing: if the model names a topic that is already done (or one
    # that does not exist), the node would re-retrieve forever and the supervisor would keep
    # routing here. An LLM decision that can cause a loop MUST be range-checked against the state
    # it claims to advance.
    topic, source = (out.topic, "llm") if (out and out.topic in todo) else (todo[0], "fallback")

    hits = search_kb(topic)
    # Keep topic/id/text: the fact-checker needs the ids, the writer needs the text.
    findings = [{"topic": topic, "id": h["id"], "text": h["text"]} for h in hits]

    return {"findings": findings,
            "log": [f"researcher[{source}]: retrieved {len(findings)} record(s) for '{topic}'"]}

probe = {"plan": ["vendor_concentration", "incident_history"], "findings": []}
out = researcher_node(probe)
print(out["log"][0])
print("ids:", [f["id"] for f in out["findings"]])

"""
EXPECTED OUTPUT  (representative)
---------------
researcher[llm]: retrieved 2 record(s) for 'vendor_concentration'
ids: ['S1', 'S2']

What must hold either way: the chosen topic is one of the not-yet-retrieved topics, so the
number of outstanding topics strictly decreases every turn.
"""

In [ ]:
# The WRITER - compose cited prose from the findings, and revise when a critic objects.
@scoped("writer")
def writer_node(state: TeamState) -> dict:
    ctx = context_for("writer", state)
    problems = ctx["problems"].get("unsupported", []) if ctx["problems"] else []
    is_revision = bool(state["draft"])

    evidence = "\n".join(f"[{f['id']}] ({f['topic']}) {f['text']}" for f in ctx["findings"])
    fix = (f"\n\nYour previous draft cited records {problems}, which do NOT exist. "
           "Remove those sentences entirely. Do not reword them, do not re-tag them."
           if problems else "")

    draft = ask(
        SYSTEM_PROMPTS["writer"] + " Use EXACTLY the section headers given, in that order. "
        "Cite every factual sentence with a [Sx] tag that appears in the evidence. Under 300 words. "
        "Output markdown only - no preamble, no code fences.",
        f"Brief:\n{ctx['brief']}\n\nRequired section headers:\n{required_headers(ctx['plan'])}"
        f"\n\nEvidence:\n{evidence}{fix}")

    source = "llm"
    if not draft:
        # The floor drops the rejected tags mechanically, which is how a template writer "revises".
        draft, source = _compose_fallback(state, drop_tags=problems), "fallback"

    return {
        "draft": draft,
        "revision_count": state["revision_count"] + (1 if is_revision else 0),
        # Any rewrite VOIDS the previous verification AND the previous approval. If you skip these
        # two lines you will eventually ship a document that was fact-checked in a version nobody
        # published - and no test will tell you, because every field looks populated.
        "fact_check": {},
        "review": {},
        "log": [f"writer[{source}]: "
                f"{'revision ' + str(state['revision_count'] + 1) if is_revision else 'draft v0'}"
                f" ({len(draft.split())} words, told to drop={problems})"],
    }

print("writer node defined - it is exercised inside the graph below.")

"""
EXPECTED OUTPUT
---------------
writer node defined - it is exercised inside the graph below.
"""

In [ ]:
# The FACT-CHECKER - the critic that makes this team worth building.
#
# This node is deliberately built the opposite way round from the others: the DETERMINISTIC check
# is the verdict, and the model runs alongside it as a second opinion whose disagreement gets
# logged. That is not distrust for its own sake. "Does this tag appear in this set?" is a question
# with one right answer, computable in three lines. Handing it to a model buys you nothing and
# costs you a verifier you cannot verify.
#
#   Rule of thumb: use a model where the answer is a JUDGEMENT (is this well structured?),
#   and use code where the answer is a FACT (is S9 in the retrieved ids?).
@scoped("fact_checker")
def fact_checker_node(state: TeamState) -> dict:
    ctx = context_for("fact_checker", state)     # draft + findings; NO brief, NO plan

    cited      = set(re.findall(r"\[(S\d+)\]", ctx["draft"]))   # every tag the draft claims
    supported  = {f["id"] for f in ctx["findings"]}             # every id we actually retrieved
    unsupported = sorted(cited - supported)                     # anything cited but not retrieved
    ok = not unsupported

    # Second opinion. Useful in real systems for the cases regex CANNOT see - a correctly-tagged
    # sentence that misrepresents what the record says. We log agreement, we do not route on it.
    note = ""
    out = ask_structured(
        SYSTEM_PROMPTS["fact_checker"],
        f"Source record ids: {sorted(supported)}\n\nDraft:\n{ctx['draft']}\n\n"
        "List the citation tags used in the draft that are NOT in the source record ids.",
        FactCheck)
    if out:
        llm_unsupported = sorted({t.lstrip("[").rstrip("]") for t in out.unsupported})
        note = (" | llm agrees" if llm_unsupported == unsupported
                else f" | llm DISAGREES (said {llm_unsupported})")

    return {"fact_check": {"ok": ok, "unsupported": unsupported},
            "log": [f"fact_checker: "
                    f"{'all citations supported' if ok else str(len(unsupported)) + ' unsupported: ' + ','.join(unsupported)}"
                    f"{note}"]}

probe = {"draft": "Claim one. [S1]\nClaim two. [S9]", "findings": [{"id": "S1"}]}
print(fact_checker_node(probe)["fact_check"])
print(fact_checker_node(probe)["log"][0])

"""
EXPECTED OUTPUT  (representative)
---------------
{'ok': False, 'unsupported': ['S9']}
fact_checker: 1 unsupported: S9 | llm agrees

The dict is IDENTICAL with or without a key - the verdict is computed, not generated.
Only the trailing note changes. If you ever see "llm DISAGREES", that is the cell doing its job.
"""

In [ ]:
# The REVIEWER (LLM-as-judge) and the ESCALATION node.
#
# Structure and completeness ARE a judgement - "does this read as a briefing an executive could
# act on?" has no regex - so here the model leads. But it leads over an objective floor: a
# deterministic scan for missing sections and length runs first and its findings are handed to the
# model as evidence. Approval requires BOTH (no objective defects AND the judge agrees), which is
# the standard shape for an LLM-as-judge you are willing to put in a control path.
@scoped("reviewer")
def reviewer_node(state: TeamState) -> dict:
    ctx = context_for("reviewer", state)          # draft + plan; it never sees the findings
    hard_notes = _structure_notes(ctx["draft"], ctx["plan"])   # objective, cheap, always runs

    out = ask_structured(
        SYSTEM_PROMPTS["reviewer"],
        f"Required section headers:\n{required_headers(ctx['plan'])}\n\n"
        f"Automated structural scan found: {hard_notes or 'no problems'}\n\n"
        f"Draft:\n{ctx['draft']}\n\n"
        "Approve only if every required section is present and the briefing is usable as-is.",
        Review)

    if out:
        # AND, not OR: the judge may add objections, never waive the objective ones.
        approved = out.approved and not hard_notes
        notes = hard_notes + [n for n in out.notes if n not in hard_notes]
        source = "llm"
    else:
        approved, notes, source = not hard_notes, hard_notes, "fallback"

    return {"review": {"approved": approved, "notes": notes},
            "log": [f"reviewer[{source}]: "
                    f"{'approved' if approved else 'changes requested: ' + '; '.join(notes)}"]}

@scoped("escalate")
def escalate_node(state: TeamState) -> dict:
    # The team could not converge within its revision budget. It stops and hands to a human rather
    # than burning tokens - Session 1's human-in-the-loop gate, reached automatically.
    return {"status": "escalated_to_human",
            "log": [f"ESCALATED after {state['revision_count']} revision(s) - human review required"]}

print(reviewer_node({"draft": "## Risks\nAll fine.", "plan": []})["review"])

"""
EXPECTED OUTPUT  (representative)
---------------
{'approved': True, 'notes': []}

With no plan the only requirement is a Risks section, which this draft has. Without a key the
fallback returns the same verdict from the structural scan alone.
"""

---
## A3 · The supervisor: the model may *route*, but it may not *authorise*

A supervisor makes one decision: **who works next?** You have two ways to implement it, and the interesting answer is "both".

* **As a pure function of state.** You get unit tests, a readable trace, and a coordination layer that behaves identically on every run.
* **As an LLM call.** You get flexibility: a supervisor that can weigh a situation you didn't enumerate.

The production shape is to **wrap the second in the first**:

1. Compute the set of **legal** routes for the current state — routes that are structurally possible *and* within budget.
2. Ask the model to choose.
3. Accept its choice **only if it is in the legal set**; otherwise fall back to the deterministic policy.

Read the ordering below as design, because it is:

1. **Plan before research.** Retrieval without a plan is unbounded.
2. **Research before writing.** A writer with no findings invents them.
3. **Fact-check before review.** The cheap, objective check runs before the expensive, subjective one. Never pay for a style review of a document that cites a source that doesn't exist.
4. **Both critics can send work back — but only within a revision budget.** Past the budget, escalate rather than loop.

Two things the model is structurally **unable** to do here, no matter what it outputs:

* **Route to `writer` once the revision budget is spent.** That route is simply not in the legal set, so `escalate` is the only way out.
* **Route to `done` while the fact-check is failing or the review is unapproved.** "Ship it" is an authorisation, and authorisations do not come from a token sampler.

> **The TA note, plainly:** a loop-back edge without a max-revision counter is not a bug you find in testing. It is a bill you find in production. Two agents can disagree politely and indefinitely.

In [ ]:
# The routing POLICY - a pure function of state. No model, no I/O, no surprises.
# SPEC (first match wins - the order IS the design):
#   1. no plan yet                           -> planner
#   2. some planned topic not yet researched  -> researcher
#   3. no draft yet                           -> writer
#   4. draft not yet fact-checked             -> fact_checker
#   5. fact check failed                      -> writer if under budget, else escalate
#   6. draft not yet reviewed                 -> reviewer
#   7. review not approved                    -> writer if under budget, else escalate
#   8. otherwise                              -> done
def supervisor_policy(state: TeamState) -> str:
    if not state["plan"]:
        return "planner"

    # Compare planned topics against topics already present in findings. Set difference, not a
    # counter: the researcher may retrieve zero records for a topic, and a counter would then
    # never advance while the set difference correctly does.
    if set(state["plan"]) - {f["topic"] for f in state["findings"]}:
        return "researcher"

    if not state["draft"]:
        return "writer"
    if not state["fact_check"]:
        return "fact_checker"

    # The loop-back, WITH the budget check. This single line is the difference between a team
    # that converges and a team that bills forever.
    if not state["fact_check"]["ok"]:
        return "writer" if state["revision_count"] < MAX_REVISIONS else "escalate"

    if not state["review"]:
        return "reviewer"
    if not state["review"]["approved"]:
        return "writer" if state["revision_count"] < MAX_REVISIONS else "escalate"

    return "done"


def legal_routes(state: TeamState) -> set[str]:
    """Every route that is STRUCTURALLY possible right now. This is the model's leash.

    The policy above always returns a member of this set, so falling back is always safe. The
    model gets to pick a different member - reordering work - but it cannot pick a non-member,
    which is what stops 'route to researcher when nothing is left to research' style loops."""
    legal = set()
    needs_rework = (state["fact_check"] and not state["fact_check"]["ok"]) or \
                   (state["review"] and not state["review"]["approved"])
    verified = bool(state["fact_check"]) and state["fact_check"]["ok"]
    approved = bool(state["review"]) and state["review"]["approved"]
    outstanding = set(state["plan"]) - {f["topic"] for f in state["findings"]}

    if not state["plan"]:
        legal.add("planner")
    if state["plan"] and outstanding:
        legal.add("researcher")
    # Writing is legal only once the evidence is in. Offering "writer" while topics are still
    # outstanding would let the model skip retrieval - and a writer with no findings invents them.
    if state["plan"] and not outstanding and not state["draft"]:
        legal.add("writer")
    if state["draft"] and needs_rework and state["revision_count"] < MAX_REVISIONS:
        legal.add("writer")                      # revising is legal only INSIDE the budget
    if state["draft"] and not state["fact_check"]:
        legal.add("fact_checker")
    if verified and not state["review"]:
        legal.add("reviewer")
    if needs_rework and state["revision_count"] >= MAX_REVISIONS:
        legal.add("escalate")                    # budget spent and work still rejected
    if verified and approved:
        legal.add("done")                        # shipping requires BOTH gates passed
    return legal or {"done"}


@scoped("supervisor")
def supervisor_node(state: TeamState) -> dict:
    policy_route = supervisor_policy(state)       # the deterministic ground truth
    legal = legal_routes(state)

    out = ask_structured(
        SYSTEM_PROMPTS["supervisor"],
        f"Current state:\n"
        f"- plan: {state['plan']}\n"
        f"- topics retrieved: {sorted({f['topic'] for f in state['findings']})}\n"
        f"- draft written: {bool(state['draft'])}\n"
        f"- fact_check: {state['fact_check'] or 'not run'}\n"
        f"- review: {state['review'] or 'not run'}\n"
        f"- revisions used: {state['revision_count']} of {MAX_REVISIONS}\n\n"
        f"Legal choices right now: {sorted(legal)}\nChoose one.",
        Route)

    # THE GUARD. An illegal route is discarded silently in favour of the policy; a legal one is
    # honoured. Nothing the model can emit lets it exceed the budget or authorise shipping,
    # because those routes are absent from `legal` in the first place.
    if out and out.next_agent in legal:
        route, source = out.next_agent, "llm"
        note = "" if route == policy_route else f" (policy would have chosen {policy_route})"
    elif out:
        # The model named something that is not legal in this state. This is not an error to
        # crash on - it is the normal, expected minority case, and the reason the guard exists.
        route, source = policy_route, "policy"
        note = f" (rejected illegal llm choice: {out.next_agent})"
    else:
        route, source = policy_route, "policy"       # no model available at all
        note = ""

    return {"next_agent": route,
            "log": [f"supervisor[{source}] -> {route}{note}"]}


base = {"brief": BRIEF, "plan": [], "findings": [], "draft": "", "fact_check": {},
        "review": {}, "revision_count": 0}
print("policy on a fresh state           :", supervisor_policy(base))
spent = {**base, "plan": ["a"], "findings": [{"topic": "a"}], "draft": "d",
         "fact_check": {"ok": False, "unsupported": ["S9"]}, "revision_count": MAX_REVISIONS}
print("legal routes with budget spent    :", sorted(legal_routes(spent)))
print("policy with budget spent          :", supervisor_policy(spent))

"""
EXPECTED OUTPUT
---------------
policy on a fresh state           : planner
legal routes with budget spent    : ['escalate']
policy with budget spent          : escalate

Read the middle line again: when the budget is gone, 'writer' is not merely discouraged,
it is not on the menu. The model cannot choose it because it is not offered.
"""

In [ ]:
# Self-check A3 - the whole routing policy, tested without a graph, a model, or a token.
# This is the payoff of writing the supervisor as a pure function: your team's coordination logic
# gets the same test coverage as any other business rule. Try doing this to a prompt.
S = lambda **kw: {**base, **kw}

cases = [
 (S(),                                                                           "planner"),
 (S(plan=["a", "b"], findings=[{"topic": "a"}]),                                 "researcher"),
 (S(plan=["a"], findings=[{"topic": "a"}]),                                      "writer"),
 (S(plan=["a"], findings=[{"topic": "a"}], draft="d"),                           "fact_checker"),
 (S(plan=["a"], findings=[{"topic": "a"}], draft="d",
    fact_check={"ok": False, "unsupported": ["S9"]}, revision_count=0),          "writer"),
 (S(plan=["a"], findings=[{"topic": "a"}], draft="d",
    fact_check={"ok": False, "unsupported": ["S9"]},
    revision_count=MAX_REVISIONS),                                               "escalate"),
 (S(plan=["a"], findings=[{"topic": "a"}], draft="d", fact_check={"ok": True}),   "reviewer"),
 (S(plan=["a"], findings=[{"topic": "a"}], draft="d", fact_check={"ok": True},
    review={"approved": False}, revision_count=MAX_REVISIONS),                    "escalate"),
 (S(plan=["a"], findings=[{"topic": "a"}], draft="d", fact_check={"ok": True},
    review={"approved": True}),                                                   "done"),
]
for state, expected in cases:
    got = supervisor_policy(state)
    assert got == expected, f"expected {expected!r}, got {got!r} for {state}"
    # and the invariant the LLM guard depends on: the policy never proposes an illegal route
    assert got in legal_routes(state), f"policy proposed an illegal route {got!r}"

print(f"PASS - all {len(cases)} routing cases, including both budget-exhausted escalations.")
print("PASS - the policy's answer is always inside legal_routes(), so the LLM fallback is safe.")

"""
EXPECTED OUTPUT
---------------
PASS - all 9 routing cases, including both budget-exhausted escalations.
PASS - the policy's answer is always inside legal_routes(), so the LLM fallback is safe.
"""

---
## A4 · The topology

Every specialist returns to the supervisor; no specialist edges to another. That constraint *is* the supervisor pattern, and it keeps the trace readable:

```
                    ┌──────────────┐
        START ─────►│  SUPERVISOR  │◄──────────────┐
                    └──────┬───────┘               │
             ┌──────┬──────┼───────┬────────┐      │
             ▼      ▼      ▼       ▼        ▼      │
          planner research writer factcheck review │
             └──────┴──────┴───────┴────────┴──────┘
                    (every agent returns to the supervisor)
                           │
                  ┌────────┴────────┐
                  ▼                 ▼
              escalate             END
```

The wiring below is one line of LangGraph worth staring at:

```python
tb.add_conditional_edges("supervisor", lambda s: s["next_agent"], {...})
```

The **node** decided (it wrote `next_agent` into state); the **edge** merely reads that decision and looks it up in a mapping. That separation — *decide in a node, route in an edge* — is what makes the decision inspectable in state history and testable in isolation. An edge function that itself calls a model is a decision you can never replay.

**A consequence to know before it bites:** every unit of work costs *two* supersteps — one for the specialist, one for the supervisor's next decision. This team runs ~19 supersteps against LangGraph's default recursion limit of 25. Star topologies hit that limit sooner than expected, and the resulting `GraphRecursionError` reads like a logic bug rather than a budget you never set. Set it explicitly.

In [ ]:
# Wire the star.
from langgraph.graph import StateGraph, START, END

SPECIALISTS = ["planner", "researcher", "writer", "fact_checker", "reviewer"]

def build_team(writer=writer_node, researcher=researcher_node):
    """Build the team graph. Parameterised on the two nodes we swap later - a hallucinating
    writer in A5, an MCP-backed researcher in Lab B - so those experiments change ONE argument
    instead of copy-pasting the topology three times."""
    tb = StateGraph(TeamState)
    tb.add_node("supervisor",   supervisor_node)
    tb.add_node("planner",      planner_node)
    tb.add_node("researcher",   researcher)
    tb.add_node("writer",       writer)
    tb.add_node("fact_checker", fact_checker_node)
    tb.add_node("reviewer",     reviewer_node)
    tb.add_node("escalate",     escalate_node)

    tb.add_edge(START, "supervisor")

    # Route FROM the supervisor. The lambda only READS the decision the supervisor already wrote
    # into state - it never decides anything itself. The third argument is a mapping from the
    # lambda's return value to a node name; "done" maps to END, which is how the graph terminates.
    tb.add_conditional_edges(
        "supervisor",
        lambda s: s["next_agent"],
        {**{a: a for a in SPECIALISTS}, "escalate": "escalate", "done": END},
    )

    # Every specialist returns to the supervisor. This one loop is the whole pattern: it is what
    # makes the supervisor a single point of coordination rather than a first step.
    for a in SPECIALISTS:
        tb.add_edge(a, "supervisor")

    tb.add_edge("escalate", END)
    return tb.compile()

team = build_team()
print("compiled nodes:", sorted(team.get_graph().nodes))
show_graph(team, "Lab A - supervisor star topology")

"""
EXPECTED OUTPUT
---------------
compiled nodes: ['__end__', '__start__', 'escalate', 'fact_checker', 'planner', 'researcher', 'reviewer', 'supervisor', 'writer']
(followed by the graph image, or Mermaid source if mermaid.ink is unavailable)
"""

In [ ]:
# Run the team and read the trajectory.
seed = {"brief": BRIEF, "plan": [], "findings": [], "draft": "", "fact_check": {},
        "review": {}, "revision_count": 0, "next_agent": "", "status": "", "log": []}

# recursion_limit is a SUPERSTEP budget, not a node-visit budget. Star topologies burn two per
# unit of work, so the default 25 is tighter than it looks. Set it deliberately.
final = team.invoke(seed, {"recursion_limit": 50})

print("TRAJECTORY")
print("----------")
for line in final["log"]:
    print(" ", line)
print(f"\nrevisions: {final['revision_count']}  |  fact_check: {final['fact_check']}"
      f"  |  status: {final['status'] or 'converged'}")
print("\n--- shipped draft ---\n")
print(final["draft"])

"""
EXPECTED OUTPUT  (representative - a live model's wording and revision count vary)
---------------
TRAJECTORY
----------
  supervisor[llm] -> planner
  planner[llm]: 3 step(s) -> ['vendor_concentration', 'incident_history', 'mitigation']
  supervisor[llm] -> researcher
  researcher[llm]: retrieved 2 record(s) for 'vendor_concentration'
  supervisor[llm] -> researcher
  researcher[llm]: retrieved 1 record(s) for 'incident_history'
  supervisor[llm] -> researcher
  researcher[llm]: retrieved 2 record(s) for 'mitigation'
  supervisor[llm] -> writer
  writer[llm]: draft v0 (168 words, told to drop=[])
  supervisor[llm] -> fact_checker
  fact_checker: all citations supported | llm agrees
  supervisor[llm] -> reviewer
  reviewer[llm]: approved
  supervisor[llm] -> done

revisions: 0  |  fact_check: {'ok': True, 'unsupported': []}  |  status: converged

--- shipped draft ---
# Vendor Concentration Risk ...

WHAT MUST HOLD regardless of wording:
  * the plan comes before any retrieval, and retrieval before the first draft;
  * fact_check runs before review;
  * revisions never exceed MAX_REVISIONS;
  * the run ends either converged (verified AND approved) or escalated.
Without a key you see [fallback] tags, one revision (the template plants a fake [S9]),
and the same shape.
"""

In [ ]:
# Self-check A4 - INVARIANTS, not a transcript.
# A live model's phrasing varies run to run; its OBLIGATIONS do not. This is how you test an LLM
# system: assert the contract, never the exact string. If you find yourself asserting on generated
# prose, you have written a test that fails on Tuesdays.
supported_ids = {f["id"] for f in final["findings"]}
cited = set(re.findall(r"\[(S\d+)\]", final["draft"]))

# 1. The planner stayed inside topics the knowledge base can actually answer.
assert set(final["plan"]) <= set(ALLOWED_TOPICS), "planner invented a topic off-list"
assert final["plan"], "the planner produced no plan at all"
# 2. Every planned topic was actually researched - one topic per turn, no gaps.
assert set(final["plan"]) <= {f["topic"] for f in final["findings"]}, "a topic went unresearched"
# 3. The revise loop respected its budget, whichever way the run ended.
assert final["revision_count"] <= MAX_REVISIONS, "the revise loop ran past its budget"
# 4. The supervisor's last act was a legal one.
assert final["next_agent"] in ("done", "escalate"), "the graph exited through an unexpected route"

if final["status"] == "":                        # the team CONVERGED
    # 5a. Convergence is only reachable through a passed fact-check AND reviewer approval...
    assert final["fact_check"].get("ok"),     "converged draft must have passed fact-check"
    assert final["review"].get("approved"),   "converged draft must be reviewer-approved"
    # 5b. ...so THE invariant holds: every citation shipped resolves to a retrieved record.
    assert cited, "a converged briefing must actually cite its evidence"
    assert cited <= supported_ids, f"unsupported citation survived: {sorted(cited - supported_ids)}"
    outcome = f"converged in {final['revision_count']} revision(s)"
else:                                            # the team ESCALATED - also a correct outcome
    assert final["status"] == "escalated_to_human", "only valid non-converged outcome"
    assert final["revision_count"] == MAX_REVISIONS, "escalation must mean the budget was spent"
    outcome = "escalated to a human (the model could not satisfy the critic within budget)"

print(f"PASS - the team upheld every invariant; {outcome}.")
print(f"Citations in the shipped draft: {sorted(cited)} - all backed by retrieved records.")

"""
EXPECTED OUTPUT  (representative)
---------------
PASS - the team upheld every invariant; converged in 0 revision(s).
Citations in the shipped draft: ['S1', 'S2', 'S3', 'S4', 'S5'] - all backed by retrieved records.
"""

---
## A5 · Why the critics exist, demonstrated on purpose

The run above may well have converged on the first attempt — a good model, given clean evidence and explicit section headers, often does. That is a pleasant result and a useless demonstration: it tells you nothing about what happens when the writer *does* invent a source, which is the failure the whole five-agent design exists to catch.

So we cause it. The next cell swaps in a **hallucinating writer** — the same node, plus one fabricated citation `[S9]` appended to whatever the model wrote. Nothing else in the team changes, and nothing else in the team is told.

Watch three things in the trace:

1. The fact-checker names `S9` without ever having seen the brief — it cannot be talked round, because it does not know what the document was trying to argue.
2. The supervisor routes back to the writer, and `revision_count` increments.
3. The fabricated tag is **absent from the shipped draft**.

That is the one thing five agents bought you that one agent would not have.

In [ ]:
# A writer that invents a source - the failure mode the team exists to catch.
@scoped("writer")
def hallucinating_writer(state: TeamState) -> dict:
    out = writer_node(state)                       # the real writer, unchanged...
    problems = (state["fact_check"] or {}).get("unsupported", [])
    if not state["draft"]:                         # ...but on the FIRST draft only, plant a fake.
        out = {**out, "draft": out["draft"] +
               "\nNorthgate Pay is the regional market leader and unlikely to fail. [S9]"}
        out["log"] = ["writer[hallucinating]: draft v0 with a fabricated [S9] citation"]
    else:
        out["log"] = [f"writer[hallucinating]: revision {state['revision_count'] + 1}, "
                      f"told to drop {problems}"]
    return out

halluc_team = build_team(writer=hallucinating_writer)
halluc = halluc_team.invoke(seed, {"recursion_limit": 50})

for line in halluc["log"]:
    print(" ", line)

# The three claims from the markdown above, as assertions.
assert any("S9" in l for l in halluc["log"]), "the fact-checker must have named S9"
assert halluc["revision_count"] >= 1, "the loop-back edge must have fired"
assert "[S9]" not in halluc["draft"], "the fabricated citation must not survive into the artefact"
assert "S9" not in {f["id"] for f in halluc["findings"]}, "S9 was never a real record"

print("\nPASS - a fabricated citation was planted, caught by a critic that could not see the")
print("brief, and removed by a revision that the supervisor routed and the budget bounded.")

"""
EXPECTED OUTPUT  (representative)
---------------
  supervisor[llm] -> planner
  planner[llm]: 3 step(s) -> ['vendor_concentration', 'incident_history', 'mitigation']
  supervisor[llm] -> researcher
  researcher[llm]: retrieved 2 record(s) for 'vendor_concentration'
  supervisor[llm] -> researcher
  researcher[llm]: retrieved 1 record(s) for 'incident_history'
  supervisor[llm] -> researcher
  researcher[llm]: retrieved 2 record(s) for 'mitigation'
  supervisor[llm] -> writer
  writer[hallucinating]: draft v0 with a fabricated [S9] citation
  supervisor[llm] -> fact_checker
  fact_checker: 1 unsupported: S9 | llm agrees
  supervisor[llm] -> writer
  writer[hallucinating]: revision 1, told to drop ['S9']
  supervisor[llm] -> fact_checker
  fact_checker: all citations supported | llm agrees
  supervisor[llm] -> reviewer
  reviewer[llm]: approved
  supervisor[llm] -> done

PASS - a fabricated citation was planted, caught by a critic that could not see the
brief, and removed by a revision that the supervisor routed and the budget bounded.
"""

In [ ]:
# And what happens when the team CANNOT converge.
# A writer that ignores feedback is not a strawman: it is what an LLM writer does when the fix is
# beyond it, or when the critic's complaint is unactionable. Without a budget, this pair loops
# until your recursion limit or your invoice stops it.
@scoped("writer")
def stubborn_writer(state: TeamState) -> dict:
    draft = _compose_fallback(state)               # re-plants [S9] every single time
    return {"draft": draft,
            "revision_count": state["revision_count"] + (1 if state["draft"] else 0),
            "fact_check": {}, "review": {},
            "log": [f"writer[stubborn]: rewrite #{state['revision_count']} (fixed nothing)"]}

stuck = build_team(writer=stubborn_writer).invoke(seed, {"recursion_limit": 50})

print("status         :", stuck["status"])
print("revision_count :", stuck["revision_count"], f"(budget was MAX_REVISIONS={MAX_REVISIONS})")
print("last log line  :", stuck["log"][-1])

assert stuck["status"] == "escalated_to_human"
assert stuck["revision_count"] == MAX_REVISIONS

print("\nThe cap converted an infinite argument into a bounded cost plus a human handoff.")
print("Note the supervisor never had the option to keep revising: once the budget was spent,")
print("'writer' left legal_routes() entirely, so neither the policy NOR the model could pick it.")

"""
EXPECTED OUTPUT
---------------
status         : escalated_to_human
revision_count : 2 (budget was MAX_REVISIONS=2)
last log line  : ESCALATED after 2 revision(s) - human review required

The cap converted an infinite argument into a bounded cost plus a human handoff.
Note the supervisor never had the option to keep revising: once the budget was spent,
'writer' left legal_routes() entirely, so neither the policy NOR the model could pick it.
"""

---
# Lab B · Model Context Protocol

## The problem MCP solves

Your team depends on `search_kb`, and that function is welded to this notebook. Give the same knowledge base to a coding assistant, a Slack bot, and a colleague's framework and you write **four integrations** — add a second data source and you write four more.

That is the **M×N problem**: M clients × N data sources = M×N bespoke connectors.

MCP makes it **M+N**. Wrap each data source once as a **server**, implement each application's **client** side once, and any client can talk to any server.

> **Analogy.** MCP is USB-C for tools. Before USB-C every device shipped its own charger — that is M×N. The port didn't make devices more capable; it made *connecting* them stop being a project. The analogy also warns you: a standard connector doesn't make a bad tool good. MCP standardises the plumbing, not the quality of what flows through it.

### The four primitives

| Primitive | Who controls it | What it is | Example here |
|---|---|---|---|
| **Tool** | The *model* decides to call it | A function with side effects or computation | `search_kb(topic)` |
| **Resource** | The *application* fetches it | Read-only context, addressed by URI | `kb://policy/citation-rules` |
| **Prompt** | The *user* chooses it | A reusable templated interaction | (not used today) |
| **Sampling** | The *server* asks the client | Server requests a model completion | (not used today) |

The split matters more than it looks: **tools are model-driven, resources are app-driven.** A resource is not "a tool that returns text" — it is context your application decides to load, with no model in the loop.

### Transports

* **stdio** — the server runs as a **child process**; messages are JSON-RPC over its stdin/stdout. Local, no ports, no auth. This is what we use.
* **streamable HTTP** — the server is a web service. Remote, multi-client, needs auth. One config key different on the client, as you'll see at the very end.

### The two libraries

| Side | Library | Why this one |
|---|---|---|
| **Server** | [`fastmcp`](https://gofastmcp.com) | Decorators (`@mcp.tool`, `@mcp.resource`) generate the JSON-RPC surface and the JSON schema from your type hints |
| **Client** | [`langchain-mcp-adapters`](https://docs.langchain.com/oss/python/langchain/mcp) | `MultiServerMCPClient` does the spawn, handshake and discovery, and hands you ordinary LangChain tools |

You will build one server and consume it three ways: directly, from a `create_agent` ReAct agent, and from the LangGraph team you just built.

### What the server exposes, and what it deliberately does not

Rather than a toy calculator, this server exposes **the project folder you are sitting in** — list files, read files, search the knowledge base, save a note. That is genuinely useful and genuinely dangerous, which makes it the right thing to practise on.

So the very first thing in the server is a **sandbox**, not a feature:

* every path is resolved and must land **inside the project directory** — `..` traversal and absolute paths are rejected;
* symlinks are refused, because a symlink is how a "safe" relative path becomes an unsafe absolute one;
* reads are size-capped;
* writes go **only** into a `mcp_workspace/` subfolder;
* **there is no delete tool, and no arbitrary-path write tool.** Not "guarded" — absent.

That last line is the design rule worth taking away: *the strongest guard rail is a capability you never exposed.* A model cannot misuse a tool that does not exist, and you never have to reason about whether your prompt talked it out of using one.

In [ ]:
%%writefile project_mcp_server.py
"""A FastMCP server exposing this project folder, safely, over stdio.

Run it standalone with:  python project_mcp_server.py
(it then sits waiting on stdin for JSON-RPC - that silence means it is working)
"""
import csv, os, sys
from pathlib import Path
from fastmcp import FastMCP

# Resolve the sandbox root from __file__, NOT from os.getcwd(). The client launches this file as a
# SUBPROCESS and the working directory of that subprocess is not guaranteed to be the notebook's.
# Relative paths are the second most common MCP setup bug after version skew.
PROJECT_ROOT = Path(__file__).resolve().parent
WORKSPACE    = PROJECT_ROOT / "mcp_workspace"       # the only writable location
CSV_PATH     = PROJECT_ROOT / "knowledge_base.csv"
MAX_READ_BYTES = 20_000

mcp = FastMCP("project-workspace")


def _safe_path(relative: str, root: Path = PROJECT_ROOT) -> Path:
    """Resolve `relative` inside `root`, or raise. THE security boundary of this server.

    Three layers, each closing a hole the previous one leaves open:
      1. resolve()          - collapses '..' and symlinks into one real absolute path, so we
                              compare the destination rather than the string the caller sent;
      2. is_relative_to()   - the destination must be inside the sandbox. This is what rejects
                              both '../../.ssh/id_rsa' and an outright absolute '/etc/passwd';
      3. is_symlink()       - a symlink INSIDE the sandbox can still point outside it, and some
                              filesystems resolve it after our check. Refuse them outright.

    Note it raises rather than returning None. Under MCP an exception becomes a protocol-level
    tool error the client sees; a silent None becomes a confusing empty result."""
    candidate = (root / relative).resolve()
    if not candidate.is_relative_to(root.resolve()):
        raise ValueError(f"path escapes the sandbox: {relative!r} is outside {root.name}/")
    if candidate.is_symlink():
        raise ValueError(f"symlinks are not allowed: {relative!r}")
    return candidate


@mcp.tool()
def list_project_files(subdir: str = ".") -> list[str]:
    """List the files in the project directory (or one of its subdirectories).

    Args:
        subdir: a path relative to the project root. Defaults to the root itself.

    Returns:
        Project-relative paths of the files directly inside that directory.
    """
    target = _safe_path(subdir)
    if not target.is_dir():
        raise ValueError(f"not a directory: {subdir!r}")
    return sorted(str(p.relative_to(PROJECT_ROOT)).replace("\\", "/")
                  for p in target.iterdir()
                  if p.is_file() and not p.name.startswith("."))


@mcp.tool()
def read_project_file(path: str) -> str:
    """Read a UTF-8 text file from inside the project directory.

    Args:
        path: a path relative to the project root, e.g. "knowledge_base.csv".

    Returns:
        The file's text, truncated to 20000 bytes.
    """
    target = _safe_path(path)
    if not target.is_file():
        raise ValueError(f"not a file: {path!r}")
    return target.read_text(encoding="utf-8", errors="replace")[:MAX_READ_BYTES]


@mcp.tool()
def search_kb(topic: str) -> list[dict]:
    """Search the internal knowledge base for records on a topic.

    Args:
        topic: one of vendor_concentration, incident_history, mitigation, contract_terms.

    Returns:
        Matching records, each with id, topic, title and text.
    """
    # ^^^ Read that docstring again. Under MCP the docstring and the type hints ARE the public API
    # contract: FastMCP turns them into the JSON schema a model uses to decide whether and how to
    # call this tool. A vague docstring is a broken integration that raises no error - it just
    # gets called wrongly, or never.
    with open(CSV_PATH, newline="", encoding="utf-8") as fh:
        return [r for r in csv.DictReader(fh) if r["topic"] == topic]


@mcp.tool()
def write_note(name: str, content: str) -> str:
    """Save a short note. Notes can only be written inside the mcp_workspace/ folder.

    Args:
        name: a bare filename such as "summary.md". Directory components are rejected.
        content: the text to write.

    Returns:
        The project-relative path that was written.
    """
    # Note the SECOND sandbox: _safe_path is called with root=WORKSPACE, so even a name that
    # somehow got past the check below could not land in the project root. Defence in depth:
    # the write capability is narrower than the read capability, on purpose.
    if "/" in name or "\\" in name or name in ("", ".", ".."):
        raise ValueError("name must be a bare filename, e.g. 'summary.md'")
    WORKSPACE.mkdir(exist_ok=True)
    target = _safe_path(name, root=WORKSPACE)
    target.write_text(content, encoding="utf-8")
    return str(target.relative_to(PROJECT_ROOT)).replace("\\", "/")


# THERE IS NO delete_file TOOL, AND NO write-to-arbitrary-path TOOL. That absence is the design.
# Adding one would mean reasoning about whether a prompt can talk the model out of using it, and
# that is a losing argument to have with a probability distribution.


@mcp.resource("kb://policy/citation-rules")
def citation_rules() -> str:
    """House rules that every generated briefing must satisfy."""
    # A RESOURCE, not a tool: the application decides to load this, no model is involved in the
    # decision. Resources are addressed by URI, and the scheme ("kb://") is yours to design -
    # treat it like a URL space, because that is exactly what it is.
    return ("CITATION POLICY\n"
            "1. Every factual sentence carries a tag of the form [S<id>].\n"
            "2. A tag is valid only if that record was retrieved for this briefing.\n"
            "3. Unsupported tags must be removed, not reworded.\n")


if __name__ == "__main__":
    # NEVER print to stdout in a stdio server. stdout IS the JSON-RPC channel; one stray print()
    # corrupts the stream and the client fails with a parse error that points nowhere near the
    # real cause. Log to stderr. (show_banner=False keeps FastMCP's own banner out of the way.)
    print("[project_mcp_server] starting on stdio", file=sys.stderr)
    mcp.run(transport="stdio", show_banner=False)

# EXPECTED OUTPUT
# ---------------
# Writing project_mcp_server.py          (or "Overwriting ..." if you re-run the cell)
#
# %%writefile only WRITES the file - it does not run it. Nothing else happens here, and that
# is correct: the client in the next cell is what starts this process. If you do want to see
# it start, run `python project_mcp_server.py` in a terminal - it will print one line to
# stderr and then sit silent, waiting on stdin. That silence is a healthy stdio server.

### Consuming it: `MultiServerMCPClient`

`MultiServerMCPClient` is the whole client side. You hand it a dict of servers; it spawns each one, runs the `initialize` handshake, asks what exists, and returns ordinary LangChain tools you can `.ainvoke()` or hand to any agent.

```python
client = MultiServerMCPClient({
    "project": {"transport": "stdio", "command": sys.executable, "args": [SERVER_PATH]},
    "weather": {"transport": "http",  "url": "https://example.com/mcp"},   # a second one, later
})
tools = await client.get_tools()
```

Two details that will save you an afternoon:

* **`command=sys.executable`, always.** A bare `"python"` resolves against the child process's `PATH`, which is often a *different* interpreter with none of your packages. The failure looks like a protocol error, not an import error.
* **Adding a second server is a dict entry.** That is the M+N claim made concrete — not "less code", but *no new integration*. The same is true for the fifth and the fiftieth.

Nothing below is hard-coded about `search_kb`. The client asks the server what it has, and the answer arrives as a JSON schema derived from your Python type hints.

In [ ]:
# Connect to the server and discover what it offers.
import sys, os, json
from langchain_mcp_adapters.client import MultiServerMCPClient

SERVER_PATH = os.path.abspath("project_mcp_server.py")

# ---------------------------------------------------------------------------------------------
# ONE PIECE OF NOTEBOOK PLUMBING, and it is worth understanding rather than copying.
#
# The stdio transport spawns the server as a real OS process and must hand the CHILD a real file
# descriptor for its stderr. Inside a Jupyter kernel `sys.stderr` is not a file - it is an
# ipykernel object that forwards text to your browser - and asking it for a file descriptor
# raises `io.UnsupportedOperation: fileno`. Run the same code in a plain .py script and it works;
# run it in a notebook and the spawn fails before the handshake even starts.
#
# The fix is to give the child a real file. You want this anyway: your MCP server's log lines
# (including the stderr message it prints on startup, and any traceback) end up in mcp_server.log
# instead of vanishing. When a stdio server misbehaves, that file is the FIRST place to look.
import sys, io
import mcp.client.stdio as _mcp_stdio
import langchain_mcp_adapters.sessions as _mcp_sessions

# 1. Open the log once, not once per cell re-run (the old version leaked a file descriptor each time).
if "MCP_SERVER_LOG" not in globals() or MCP_SERVER_LOG.closed:
    MCP_SERVER_LOG = open("mcp_server.log", "a", encoding="utf-8")

# 2. Recover the true original. A patched function carries a marker, so we can unwrap it
#    instead of trusting whatever is currently bound to the module attribute.
_current = _mcp_stdio.stdio_client
_true_original = getattr(_current, "__pre_patch__", None) or _current
if getattr(_true_original, "__module__", "") == "__main__":
    raise RuntimeError("stdio_client is already patched by an unmarked patch — restart the kernel.")
_mcp_stdio._pre_patch_stdio_client = _true_original

# 3. Bind `original` and `log` as keyword-only DEFAULTS -> evaluated once, at def time.
#    They can never be re-resolved through the notebook's global namespace.
def _stdio_client_logging_to_file(server, errlog=None, *, _orig=_true_original, _log=MCP_SERVER_LOG):
    return _orig(server, errlog if errlog is not None else _log)

_stdio_client_logging_to_file.__pre_patch__ = _true_original   # makes re-runs safe forever

_mcp_stdio.stdio_client = _stdio_client_logging_to_file
_mcp_sessions.stdio_client = _stdio_client_logging_to_file
# ---------------------------------------------------------------------------------------------

mcp_client = MultiServerMCPClient({
    "project": {
        "transport": "stdio",           # spawn it as a child process and talk over stdin/stdout
        "command": sys.executable,      # THIS interpreter - see the note above
        "args": [SERVER_PATH],
    }
})

# get_tools() does the whole dance: spawn -> initialize (version handshake) -> list_tools ->
# wrap each result as a LangChain StructuredTool. Everything you print below was DISCOVERED;
# none of it was written on this side.
mcp_tools = await mcp_client.get_tools()
TOOLS_BY_NAME = {t.name: t for t in mcp_tools}

print("DISCOVERED TOOLS (the client was told nothing in advance)")
print("-------------------------------------------------------")
for t in mcp_tools:
    print(f"  {t.name}{tuple(t.args)}")
    print(f"      {t.description.splitlines()[0]}")


def parse_tool_result(raw):
    """MCP tool results arrive as CONTENT BLOCKS, not plain Python values.

    A tool that returns list[dict] comes back as [{'type': 'text', 'text': '<json>'}] - because
    the protocol carries content, not Python objects. Different adapter/SDK versions also hand
    you a bare string or an already-parsed structure, so this small defensive helper normalises
    all three. Every real MCP project ends up writing it; here it is once, up front."""
    if isinstance(raw, list) and raw and isinstance(raw[0], dict) and "text" in raw[0]:
        raw = "".join(b.get("text", "") for b in raw if b.get("type") == "text")
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return raw
    return raw


hits = parse_tool_result(await TOOLS_BY_NAME["search_kb"].ainvoke({"topic": "mitigation"}))
print("\nsearch_kb('mitigation') ->", [r["id"] for r in hits])

files = parse_tool_result(await TOOLS_BY_NAME["list_project_files"].ainvoke({"subdir": "."}))
print("project files (first 5)  ->", files[:5])

print("\nJSON schema FastMCP derived from your type hints, for search_kb:")
print(" ", json.dumps(TOOLS_BY_NAME["search_kb"].args, indent=2).replace("\n", "\n  "))

"""
EXPECTED OUTPUT  (the file list depends on your folder)
---------------
DISCOVERED TOOLS (the client was told nothing in advance)
-------------------------------------------------------
  list_project_files('subdir',)
      List the files in the project directory (or one of its subdirectories).
  read_project_file('path',)
      Read a UTF-8 text file from inside the project directory.
  search_kb('topic',)
      Search the internal knowledge base for records on a topic.
  write_note('name', 'content')
      Save a short note. Notes can only be written inside the mcp_workspace/ folder.

search_kb('mitigation') -> ['S4', 'S5']
project files (first 5)  -> ['Day3_Session1_...ipynb', ...]

JSON schema FastMCP derived from your type hints, for search_kb:
  {
    "topic": {
      "type": "string",
      "description": "one of vendor_concentration, incident_history, mitigation, contract_terms."
    }
  }
    ^ that description came from your Args: block. The docstring IS the interface.
"""

In [ ]:
# Self-check - the sandbox has to actually hold, because an agent will eventually test it for you.
#
# One protocol behaviour to learn here: when a tool raises on the SERVER, MCP does not blow up the
# client. The error comes back as an ordinary result flagged as an error, and the LangChain
# adapter hands it to you as text. That is deliberate - the whole point is that a MODEL receives
# it, reads "path escapes the sandbox", and tries something else. An exception would kill the
# agent loop; a returned error lets it recover. So "did it refuse?" is a question about the
# RESULT, not about whether Python raised.
REFUSAL_MARKERS = ("Error calling tool", "escapes the sandbox",
                   "not allowed", "must be a bare filename", "not a directory", "not a file")

async def attempt(tool_name: str, args: dict) -> str:
    """Return the refusal message, or "" if the call was allowed through."""
    try:
        raw = await TOOLS_BY_NAME[tool_name].ainvoke(args)
    except Exception as e:                        # some adapter versions raise instead - handle both
        return f"{type(e).__name__}: {e}"
    text = raw if isinstance(raw, str) else " ".join(
        b.get("text", "") for b in raw if isinstance(b, dict))   # flatten the content blocks
    return text if any(m in text for m in REFUSAL_MARKERS) else ""

attacks = {
    "parent traversal":  ("read_project_file", {"path": "../../../etc/passwd"}),
    "absolute path":     ("read_project_file", {"path": os.path.abspath(os.sep)}),
    "dotdot in listing": ("list_project_files", {"subdir": ".."}),
    "write outside box": ("write_note",        {"name": "../escaped.md", "content": "x"}),
}
for label, (tool, args) in attacks.items():
    err = await attempt(tool, args)
    assert err, f"SANDBOX HOLE: {label} was NOT refused"
    print(f"  refused  {label:<18} -> {err.split(': ', 1)[-1][:78]}")

# ...and the legitimate paths still work, which is the other half of a useful sandbox.
ok_text = parse_tool_result(await TOOLS_BY_NAME["read_project_file"].ainvoke(
    {"path": "knowledge_base.csv"}))
assert "vendor_concentration" in ok_text, "a legitimate read must still succeed"
written = parse_tool_result(await TOOLS_BY_NAME["write_note"].ainvoke(
    {"name": "scratch.md", "content": "written through MCP"}))
assert written.startswith("mcp_workspace/"), "writes must land in the workspace folder"

print(f"\n  allowed  read knowledge_base.csv -> {len(ok_text)} chars")
print(f"  allowed  write_note              -> {written}")
print("\nPASS - four escape attempts refused, both legitimate operations allowed.")
print("The guard is in the SERVER, not the prompt. A prompt-level rule is a request;")
print("a resolved-path check is a boundary.")

"""
EXPECTED OUTPUT
---------------
  refused  parent traversal   -> '../../../etc/passwd' is outside IndustryProfessionalsAgenticAICourse/
  refused  absolute path      -> 'C:\\' is outside IndustryProfessionalsAgenticAICourse/
  refused  dotdot in listing  -> '..' is outside IndustryProfessionalsAgenticAICourse/
  refused  write outside box  -> ... name must be a bare filename, e.g. 'summary.md'

  allowed  read knowledge_base.csv -> 704 chars
  allowed  write_note              -> mcp_workspace/scratch.md

PASS - four escape attempts refused, both legitimate operations allowed.
The guard is in the SERVER, not the prompt. A prompt-level rule is a request;
a resolved-path check is a boundary.
"""

### An agent that chooses the tools itself

So far *you* called the tools. Now hand them to an agent and let the model decide. `create_agent(model, tools)` builds LangChain's prebuilt tool-calling (ReAct) agent — a small LangGraph under the hood: call the model, if it emitted tool calls run them and loop, otherwise stop.

```python
from langchain.agents import create_agent
agent = create_agent(chat_model, mcp_tools)
result = await agent.ainvoke({"messages": [{"role": "user", "content": "..."}]})
```

Two things to notice in the trace it prints:

* **Nobody told the agent that `search_kb` exists.** It appeared through MCP discovery, and the model chose it from the schema FastMCP generated out of your docstring. This is the moment the docstring stops being documentation and starts being an interface.
* **The agent may chain calls** — list the files, then read one, then search. Each hop is a separate model turn, which is exactly why an unbounded ReAct agent over a large tool set gets expensive: you pay for the whole message history on every hop.

Compare this with Lab A. Here the model chooses *what to do*; there, the graph did, and the model only chose *how to do each step*. Neither is better. The agent is more flexible and less predictable; the graph is more predictable and less flexible. Choosing between them is the whole of Session 1.

In [ ]:
# Let a real agent loose on the MCP tools.
from langchain.agents import create_agent

if LLM_ENABLED:
    # create_agent needs a chat model that supports .bind_tools() - which is exactly why we used
    # ChatLiteLLM rather than calling litellm.completion() directly.
    mcp_agent = create_agent(chat_model, mcp_tools)

    result = await mcp_agent.ainvoke({"messages": [{"role": "user", "content":
        "Using your tools, find out what the knowledge base says about mitigation options, "
        "and reply with the record ids and a one-sentence summary."}]})

    print("AGENT TRACE")
    print("-----------")
    for m in result["messages"]:
        kind = type(m).__name__.replace("Message", "")
        calls = getattr(m, "tool_calls", None)
        if calls:
            for c in calls:
                print(f"  [{kind}] tool_call: {c['name']}({c['args']})")
        elif m.content:
            text = m.content if isinstance(m.content, str) else str(m.content)
            print(f"  [{kind}] {text[:220]}")
else:
    print("Skipped - no model available. Re-run with a valid key to watch an agent pick MCP tools.")
    print("Everything else in Lab B runs without a model, because the protocol does not need one.")

"""
EXPECTED OUTPUT  (representative - the agent may take a different number of hops)
---------------
AGENT TRACE
-----------
  [Human] Using your tools, find out what the knowledge base says about mitigation options, ...
  [AI] tool_call: search_kb({'topic': 'mitigation'})
  [Tool] [{"id":"S4","topic":"mitigation","title":"Failover design note", ...}]
  [AI] Records S4 and S5: a secondary processor can sit behind the existing payment
       abstraction in about six weeks, at roughly forty thousand per year to keep warm.
"""

### B4 · Stateful sessions, and what the adapter is doing for you

`await client.get_tools()` is convenient and slightly wasteful: for a stdio server it opens a connection, discovers, and closes — and each subsequent tool call opens its own short-lived session. For a local subprocess and a handful of calls that is fine, and it is why the code above is so short.

When it stops being fine, you hold **one session open**:

```python
from langchain_mcp_adapters.tools import load_mcp_tools

async with mcp_client.session("project") as session:
    tools = await load_mcp_tools(session)     # tools bound to THIS live session
    result = await tools[0].ainvoke({...})     # every call reuses the same connection
```

Reach for a persistent session when:

| Situation | Why a per-call session breaks it |
|---|---|
| The server keeps **state between calls** — an open file handle, a cursor, a transaction, a logged-in browser | Reconnecting resets it |
| The server uses **sampling** (it asks *your* client for a model completion) or **elicitation** (it asks the user a question) | Those are callbacks on a live session; there is nobody to call back after it closes |
| The server pushes **notifications** — resource-changed, progress updates | Nothing is listening between calls |
| Connection setup is **expensive** — process spawn, TLS, auth, a cold container | You pay it on every single tool call |

Underneath both paths is the low-level `mcp` SDK, and it is worth seeing the handshake once so the adapter stops being magic:

1. **spawn** the server as a subprocess over stdio;
2. **`initialize`** — protocol version negotiation. This is where mismatched SDK versions fail, loudly and unhelpfully;
3. **`list_tools`** — discovery: the client learns what exists, having been told nothing in advance;
4. **`call_tool`** — invocation;
5. **`list_resources` / `read_resource`** — application-driven context, no model in the loop.

Step 3 is the whole idea. Step 5 is where you pick up the citation policy the writer is supposed to follow — your application decides to load it; no agent chose to.

In [ ]:
# One long-lived session, and the resource fetch that no model asked for.
from langchain_mcp_adapters.tools import load_mcp_tools

async with mcp_client.session("project") as session:
    # Inside this block `session` is a live mcp.ClientSession - the SAME object the low-level SDK
    # gives you. The adapter did the spawn and the initialize handshake; from here you can use
    # either API. load_mcp_tools() binds LangChain tools to THIS connection...
    session_tools = await load_mcp_tools(session)
    print("tools bound to the live session:", [t.name for t in session_tools])

    # ...and the raw protocol calls are still right there.
    listed = await session.list_tools()
    print("raw list_tools ->", [t.name for t in listed.tools])

    called = await session.call_tool("search_kb", {"topic": "contract_terms"})
    print("raw call_tool  ->", [r["id"] for r in json.loads(called.content[0].text)])

    # RESOURCES are application-driven. Note what did NOT happen here: no model decided to fetch
    # this. Your code did, because your code knows the writer needs the citation policy.
    resources = await session.list_resources()
    print("resources      ->", [str(r.uri) for r in resources.resources])

    policy = await session.read_resource("kb://policy/citation-rules")
    CITATION_POLICY = policy.contents[0].text

print("\nCITATION POLICY (fetched as a resource, not called as a tool)")
print("------------------------------------------------------------")
print(CITATION_POLICY)
print("Every call above reused ONE connection. Outside the `async with`, it is closed and any")
print("server-side state it held is gone - which is precisely why the distinction matters.")

"""
EXPECTED OUTPUT
---------------
tools bound to the live session: ['list_project_files', 'read_project_file', 'search_kb', 'write_note']
raw list_tools -> ['list_project_files', 'read_project_file', 'search_kb', 'write_note']
raw call_tool  -> ['S6']
resources      -> ['kb://policy/citation-rules']

CITATION POLICY (fetched as a resource, not called as a tool)
------------------------------------------------------------
CITATION POLICY
1. Every factual sentence carries a tag of the form [S<id>].
2. A tag is valid only if that record was retrieved for this briefing.
3. Unsupported tags must be removed, not reworded.

Every call above reused ONE connection. Outside the `async with`, it is closed and any
server-side state it held is gone - which is precisely why the distinction matters.
"""

---
## B5 · Bring it together — Milestone 6

Swap the team's Researcher from the local `search_kb` function to the MCP tool. Everything else — state, scopes, supervisor policy, loop-backs, the cap, both critics — stays untouched.

Two mechanical notes:

1. **The node becomes `async`**, because the MCP call is awaited. LangGraph supports async nodes; invoke the graph with `await team.ainvoke(...)` instead of `team.invoke(...)`. (The `scoped()` decorator you wrote in A1 already handles this — it checks `inspect.iscoroutinefunction` and wraps accordingly. A decorator that only understood sync functions would silently wrap the coroutine and fail somewhere unrecognisable.)
2. **Nothing else changes.** That is the deliverable. If swapping your data access forced you to touch your supervisor, your layering was wrong.

Then the sharper question: **what did MCP actually buy?** Latency went *up* — every call now crosses a process boundary and gets serialised to JSON. What you gained is that the knowledge base is no longer welded to this notebook: the agent you built two cells ago, Claude Desktop, and a colleague's framework all reach it through the same server, with zero changes to it. If that's worth nothing for a given tool, a plain Python function is still the right answer. **MCP is an integration decision, not an upgrade.**

In [ ]:
# The MCP-backed Researcher. Same scope, same contract, different transport.
@scoped("researcher")
async def mcp_researcher_node(state: TeamState) -> dict:
    ctx = context_for("researcher", state)
    todo = [t for t in ctx["plan"] if t not in ctx["already"]]

    out = ask_structured(
        SYSTEM_PROMPTS["researcher"],
        f"Topics not yet retrieved: {todo}\nChoose exactly one to retrieve now.",
        NextTopic)
    topic, source = (out.topic, "llm") if (out and out.topic in todo) else (todo[0], "fallback")

    # THE one changed line: retrieval crosses a process boundary instead of a function call.
    raw = await TOOLS_BY_NAME["search_kb"].ainvoke({"topic": topic})
    hits = parse_tool_result(raw)

    findings = [{"topic": topic, "id": h["id"], "text": h["text"]} for h in hits]
    return {"findings": findings,
            "log": [f"researcher[{source}, MCP]: retrieved {len(findings)} record(s) for '{topic}'"]}

# Rebuild with exactly one node swapped. Nothing else in the team is aware of the change - which
# is the actual test of whether your layering held.
mcp_team = build_team(researcher=mcp_researcher_node)

mcp_final = await mcp_team.ainvoke(seed, {"recursion_limit": 50})   # ainvoke: the graph is async now

for line in mcp_final["log"]:
    print(" ", line)
print(f"\nstatus: {mcp_final['status'] or 'converged'}  |  revisions: {mcp_final['revision_count']}")

"""
EXPECTED OUTPUT  (representative)
---------------
  supervisor[llm] -> planner
  planner[llm]: 3 step(s) -> ['vendor_concentration', 'incident_history', 'mitigation']
  supervisor[llm] -> researcher
  researcher[llm, MCP]: retrieved 2 record(s) for 'vendor_concentration'
  supervisor[llm] -> researcher
  researcher[llm, MCP]: retrieved 1 record(s) for 'incident_history'
  supervisor[llm] -> researcher
  researcher[llm, MCP]: retrieved 2 record(s) for 'mitigation'
  supervisor[llm] -> writer
  writer[llm]: draft v0 (171 words, told to drop=[])
  supervisor[llm] -> fact_checker
  fact_checker: all citations supported | llm agrees
  supervisor[llm] -> reviewer
  reviewer[llm]: approved
  supervisor[llm] -> done

status: converged  |  revisions: 0
"""

In [ ]:
# Self-check - the same team, the same evidence, a different transport.
# Note what is asserted and what is NOT. The two runs retrieved the SAME records, because
# retrieval is deterministic on both sides. The two DRAFTS may differ word for word, because a
# model wrote them - so asserting on the draft text would be asserting on the weather.
mcp_ids   = sorted(f["id"] for f in mcp_final["findings"])
local_ids = sorted(f["id"] for f in final["findings"])

assert set(mcp_final["plan"]) <= set(ALLOWED_TOPICS),  "planner stayed on-list"
assert set(mcp_final["plan"]) <= {f["topic"] for f in mcp_final["findings"]}, "a topic went unresearched"
assert mcp_ids == local_ids, f"MCP retrieval returned different evidence: {mcp_ids} vs {local_ids}"
assert mcp_final["revision_count"] <= MAX_REVISIONS, "the revise loop ran past its budget"
assert any("MCP" in l for l in mcp_final["log"]), "the team must actually have used the MCP tool"

cited = set(re.findall(r"\[(S\d+)\]", mcp_final["draft"]))
if mcp_final["status"] == "":
    assert mcp_final["fact_check"].get("ok") and mcp_final["review"].get("approved")
    assert cited <= set(mcp_ids), f"unsupported citation survived: {sorted(cited - set(mcp_ids))}"
    outcome = f"converged in {mcp_final['revision_count']} revision(s)"
else:
    assert mcp_final["status"] == "escalated_to_human"
    outcome = "escalated to a human within budget"

print(f"PASS - identical evidence over a different transport; {outcome}.")
print(f"  local function -> {local_ids}")
print(f"  MCP tool call  -> {mcp_ids}")
print("\nExactly one node changed. The supervisor policy, the scopes, the revision cap and both")
print("critics were untouched.")

"""
EXPECTED OUTPUT  (representative)
---------------
PASS - identical evidence over a different transport; converged in 0 revision(s).
  local function -> ['S1', 'S2', 'S3', 'S4', 'S5']
  MCP tool call  -> ['S1', 'S2', 'S3', 'S4', 'S5']

Exactly one node changed. The supervisor policy, the scopes, the revision cap and both
critics were untouched.
"""

### Look at what you built

Two pictures, because there are two different graphs here and confusing them is a common source of muddled mental models:

1. **The agent graph** — the LangGraph state machine. Nodes are agents, edges are routing. This is what `recursion_limit` counts and what `next_agent` drives.
2. **The MCP topology** — processes and transports. The notebook is a *client*; the server is a separate OS process; JSON-RPC flows over its stdin/stdout. None of this appears in the agent graph, because to LangGraph the researcher is just a node that takes a while.

In [ ]:
# 1. The agent graph - the state machine LangGraph actually executes.
show_graph(mcp_team, "Milestone 6 - supervisor team, researcher backed by MCP")

# 2. The MCP topology - processes and transports. Hand-written Mermaid, because this layer is
#    invisible to LangGraph: to the graph, `researcher` is simply a node that awaits something.
display(Mermaid("""

flowchart LR
    subgraph NB["this notebook (MCP client process)"]
        G["LangGraph team<br/>supervisor + 5 agents"]
        A["create_agent<br/>ReAct agent"]
        C["MultiServerMCPClient"]
        G -- "researcher node awaits" --> C
        A -- "model picks a tool" --> C
    end
    C == "JSON-RPC over stdio<br/>(initialize, list_tools, call_tool)" ==> S
    subgraph SP["project_mcp_server.py (child process)"]
        S["FastMCP 'project-workspace'"]
        S --> T1["search_kb"]
        S --> T2["read_project_file"]
        S --> T3["list_project_files"]
        S --> T4["write_note"]
        S --> R1(["resource<br/>kb://policy/citation-rules"])
    end
    T1 & T2 & T3 --> SB{{"_safe_path()<br/>sandbox boundary"}}
    T4 --> SB
    SB --> FS[("project folder<br/>knowledge_base.csv, mcp_workspace/")]

"""))

print("Two clients, one server, one sandbox. The left box is replaceable (Claude Desktop, an IDE,")
print("a colleague's framework); the right box does not change when you replace it.")

"""
EXPECTED OUTPUT
---------------
Milestone 6 - supervisor team, researcher backed by MCP
------------------------------------------------------
(the LangGraph star topology image - supervisor at the centre, five specialists, escalate, END)
(then the MCP topology diagram)

Two clients, one server, one sandbox. The left box is replaceable (Claude Desktop, an IDE,
a colleague's framework); the right box does not change when you replace it.
"""

In [ ]:
# Milestone 6 checklist for your capstone.
checklist = {
 "specialised agents with enforced write scopes":   len(AGENT_SCOPES) >= 5,
 "supervisor routing as a pure, tested function":   supervisor_policy(
     {**base, "plan": ["a"], "findings": [{"topic": "a"}]}) == "writer",
 "LLM routing constrained to legal routes":         "writer" not in legal_routes(
     {**base, "plan": ["a"], "findings": [{"topic": "a"}], "draft": "d",
      "fact_check": {"ok": False}, "revision_count": MAX_REVISIONS}),
 "at least one critic independent of the producer": "draft" not in AGENT_SCOPES["fact_checker"],
 "loop-back edge WITH a revision cap":              supervisor_policy(
     {**base, "plan": ["a"], "findings": [{"topic": "a"}], "draft": "d",
      "fact_check": {"ok": False}, "revision_count": MAX_REVISIONS}) == "escalate",
 "a fabricated citation is caught and removed":     "[S9]" not in halluc["draft"]
                                                     and halluc["revision_count"] >= 1,
 "escalation path to a human":                      stuck["status"] == "escalated_to_human",
 "custom MCP server with tools AND a resource":     len(mcp_tools) >= 3 and bool(CITATION_POLICY),
 "the server sandbox rejects path escapes":         True,   # asserted four ways above
 "server consumed from more than one client":       set(TOOLS_BY_NAME) == {t.name for t in session_tools},
 "agent team wired to the MCP tool":                any("MCP" in l for l in mcp_final["log"]),
}
for item, ok in checklist.items():
    print(f"  [{'x' if ok else ' '}] {item}")
assert all(checklist.values()), "one or more milestone criteria are not met"
print("\nPASS - this is a valid Milestone 6 skeleton.")

"""
EXPECTED OUTPUT
---------------
  [x] specialised agents with enforced write scopes
  [x] supervisor routing as a pure, tested function
  [x] LLM routing constrained to legal routes
  [x] at least one critic independent of the producer
  [x] loop-back edge WITH a revision cap
  [x] a fabricated citation is caught and removed
  [x] escalation path to a human
  [x] custom MCP server with tools AND a resource
  [x] the server sandbox rejects path escapes
  [x] server consumed from more than one client
  [x] agent team wired to the MCP tool

PASS - this is a valid Milestone 6 skeleton.
"""

---
# Appendix Ax1 · You can also consume servers other people wrote

Everything above was your server. The other half of M+N is that **you rarely have to write one**. A growing set of public MCP servers is already out there — GitHub, filesystem, browser automation, documentation search — and consuming one is the same three lines you already know, with a different transport.

```python
client = MultiServerMCPClient({
    "project":  {"transport": "stdio", "command": sys.executable, "args": [SERVER_PATH]},
    "deepwiki": {"transport": "http",  "url": "https://mcp.deepwiki.com/mcp"},   # <- one entry
})
tools = await client.get_tools()          # now returns YOUR tools AND DeepWiki's
agent = create_agent(chat_model, tools)   # the agent doesn't know or care which is which
```

That is the claim, concretely: a second data source cost one dictionary entry, not an integration project.

A few that need no signup at the time of writing:

| Server | URL | What it gives you |
|---|---|---|
| DeepWiki | `https://mcp.deepwiki.com/mcp` | Ask questions about any public GitHub repository |
| Context7 | `https://mcp.context7.com/mcp` | Up-to-date library documentation lookup |
| Hugging Face | `https://huggingface.co/mcp` | Search models, datasets and Spaces |

**Treat this appendix as illustrative, not as coursework.** Public endpoints move, rate-limit and go down; nothing else in this notebook depends on it. The cell below is wrapped so a failure prints a note and moves on — and a failure here is a fact about somebody else's uptime, not about your code.

> **The security point, since it now applies to code you didn't write:** a third-party MCP server sees every argument your agent sends it and returns text that goes straight into your model's context. Tool descriptions from an untrusted server are untrusted input. Pin versions, read what you connect to, and don't route secrets through a tool you haven't audited.

In [ ]:
# OPTIONAL - a public MCP server. Nothing else depends on this cell.
async def try_public_server():
    try:
        public = MultiServerMCPClient({
            "deepwiki": {"transport": "http", "url": "https://mcp.deepwiki.com/mcp"},
        })
        tools = await public.get_tools()
        print("DeepWiki exposed:", [t.name for t in tools])
        print("first tool schema:", list(tools[0].args))
        return True
    except Exception as e:
        print(f"Public server unreachable ({type(e).__name__}: {str(e)[:100]})")
        print("Expected - public endpoints move, rate-limit and block corporate networks.")
        print("The point stands regardless: it was ONE dict entry, and the transport key is the")
        print("only line that differs from your local stdio server.")
        return False

await try_public_server()

"""
EXPECTED OUTPUT  (one of these two - both are fine)
---------------
DeepWiki exposed: ['ask_question', 'read_wiki_contents', 'read_wiki_structure']
first tool schema: ['repoName', 'question']

  ...or, on a blocked/offline network:

Public server unreachable (ConnectError: ...)
Expected - public endpoints move, rate-limit and block corporate networks.
The point stands regardless: it was ONE dict entry, and the transport key is the
only line that differs from your local stdio server.
"""

---
# Appendix Ax2 · A2A v1.0.1 and AP2 v0.2

**Specification target:** A2A **v1.0.1** and AP2 **v0.2.0**  
**Reviewed:** 8 August 2026

| Protocol | Boundary | Question it answers | Used in this scenario for |
|---|---|---|---|
| **MCP** | agent → tool or resource | “How does an agent invoke a capability?” | The tools used elsewhere in this notebook |
| **A2A** | agent → remote agent | “How does one opaque agent delegate work to another?” | Asking a vendor agent for a report quote |
| **AP2** | security inside a commerce/payment flow | “What evidence proves the user authorised this agent-performed purchase?” | Enforcing the report budget and producing receipts |

A2A and AP2 solve different problems. **AP2 does not require A2A, and A2A does not require AP2.** This appendix composes them in one story: A2A obtains a structured quote; a commerce flow then uses AP2-style evidence to decide whether that quote may be paid.

Some reading on this:
1. [A2A Reading](https://a2a-protocol.org/latest/specification/)
2. [AP2 Readng](https://agentpaymentsprotocol.info/)

You can also refer to these for sample code on how to implement these:
1. [A2A Samples](https://github.com/a2aproject/a2a-samples)
2. [AP2 Samples in Python](https://github.com/google-agentic-commerce/AP2/tree/main/code/samples/python)


## The scenario

Your Lab A research team needs a paywalled Q3 2026 EU semiconductor-demand report.

1. The report is sold by a **vendor agent you did not build**. You cannot import its graph, prompt or memory. You discover it and request a quote over A2A.
2. The user permits the shopping agent to buy that report only from an allowed merchant, with an allowed payment instrument, before an expiry time and below a fixed amount. AP2 carries and verifies that authority.

> **Nothing to obtain or configure:** every URL, identity and key below is created locally for this simulation. The `.example` URLs are reserved examples and cannot be contacted. The code generates fresh, in-memory P-256 keypairs when you run it. There is no `AP2_USER_KEY`, `AP2_MERCHANT_KEY`, API key, dashboard value or `.env` entry for this appendix.

### What is real—and what is simplified

- The A2A objects use the current v1.0 wire shape for the fields demonstrated here.
- The AP2 claim names, `vct` values, open/closed mandate split, constraints, bindings, minor-unit money and receipt fields follow AP2 v0.2.
- Network calls, enrollment, trust registries, payment rails, selective disclosure and full SD-JWT/KB-SD-JWT processing are **not** implemented.
- The local signature wrapper uses real ECDSA P-256 signatures. A separately labelled `demo_binding_token` makes the open→closed link testable; in AP2 that job belongs to the SD-JWT/KB-SD-JWT credential chain and `sd_hash`, not to a field named `demo_binding_token`.
- These teaching wrappers are **not an AP2 credential implementation**. They keep tamper evidence executable without pretending that a notebook is a payment network.


## A2A — delegate to an agent you did not build

A2A is an open Linux Foundation protocol for communication between independent, potentially opaque agents. A2A v1.0 became stable in March 2026; v1.0.1 followed in May 2026.

### Five terms before the code

| Term | Meaning in this scenario |
|---|---|
| **Client agent** | Your research/shopping agent |
| **Remote agent** | The vendor’s market-data agent |
| **Agent Card** | The remote agent’s published capabilities, interfaces and authentication requirements |
| **Message** | One unit of communication from client (`ROLE_USER`) or remote agent (`ROLE_AGENT`) |
| **Task** | A server-created, stateful unit of work with a lifecycle; its outputs are **Artifacts** |

### The current lifecycle

1. Fetch `GET /.well-known/agent-card.json`.
2. Select one entry from `supportedInterfaces` and check its `protocolBinding` and `protocolVersion`.
3. Resolve any authentication declared in `securitySchemes` / `securityRequirements`. The remote operator—not A2A itself—provides or brokers that credential.
4. Send `SendMessage`. The response contains **exactly one** of `result.task` or `result.message` in the JSON-RPC binding.
5. If a Task is returned, poll with `GetTask`, stream, or subscribe when the advertised capabilities support it.
6. Carry identifiers deliberately:
   - `contextId` groups related Tasks and Messages into one conversation.
   - `taskId` identifies one stateful unit of work.
   - same `taskId` (+ matching `contextId`) continues that Task;
   - same `contextId`, no `taskId`, asks for a **new Task in the same conversation**.

In A2A v1 JSON-RPC, method names are PascalCase (`SendMessage`, `GetTask`). Roles and states use enum names such as `ROLE_USER` and `TASK_STATE_COMPLETED`. A `Part` is flattened—`{"text":"..."}` or `{"data":{...}}`, not a legacy `kind` wrapper.

Agent Card skills are descriptive capabilities, not RPC methods. The client does not “call skill ID `market_report_quote`”; it sends a Message. The keyword routing inside the local mock is only a small stand-in for the vendor agent's private implementation.

> **Version-sensitive integration note:** an adapter may still document a legacy A2A binding such as `message/send`. Treat an adapter’s contract separately from the v1.0.1 wire example below. Confirm its supported protocol version instead of silently mixing shapes.

Official references: [A2A specification](https://a2a-protocol.org/latest/specification/) · [v1.0 changes](https://a2a-protocol.org/latest/whats-new-v1/) · [releases](https://github.com/a2aproject/A2A/releases)


In [ ]:
# A2A v1.0.1-shaped offline fixture.
# No HTTP request is made. DEMO_ONLY_* values are ordinary local test data—not credentials.
import json
import uuid

DEMO_ONLY_MARKET_AGENT_CARD = {
    "name": "MarketData Research Agent",
    "description": "Provides paid sector-report quotes and fulfillment status.",
    "supportedInterfaces": [{
        "url": "https://agents.marketdata.example/a2a/rpc",
        "protocolBinding": "JSONRPC",
        "protocolVersion": "1.0",
    }],
    "version": "1.0.0",
    "capabilities": {
        "streaming": False,
        "pushNotifications": False,
        "extendedAgentCard": False,
    },
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain", "application/json"],
    "skills": [{
        "id": "market_report_quote",
        "name": "Market-report quote",
        "description": "Returns a structured quote for an available sector report.",
        "tags": ["research", "market-data", "paid"],
        "examples": ["Quote the Q3 2026 EU semiconductor demand report."],
    }],
}

DEMO_ONLY_MARKET_MERCHANT = {
    "id": "merchant_marketdata_demo",
    "name": "MarketData Demo Merchant",
    "website": "https://marketdata.example",
}
DEMO_ONLY_REPORT_SKU = "MD-SEMI-EU-Q3"
DEMO_ONLY_REPORT_PRICE_MINOR = 24_000  # USD 240.00; AP2 uses integer minor units.

def demo_uuid() -> str:
    return str(uuid.uuid4())

def a2a_teaching_check_card(card: dict) -> None:
    """Small invariant check for fields used here—not a replacement for the official SDK."""
    required = {"name", "description", "supportedInterfaces", "version", "capabilities",
                "defaultInputModes", "defaultOutputModes", "skills"}
    assert required <= card.keys(), f"Agent Card missing: {required - card.keys()}"
    assert card["supportedInterfaces"], "at least one AgentInterface is required"
    preferred = card["supportedInterfaces"][0]
    assert {"url", "protocolBinding", "protocolVersion"} <= preferred.keys()
    assert preferred["protocolBinding"] in {"JSONRPC", "GRPC", "HTTP+JSON"}
    assert preferred["protocolVersion"] == "1.0"
    assert card["skills"] and all({"id", "name", "description", "tags"} <= s.keys()
                                  for s in card["skills"])

def a2a_build_send_message(text: str, *, context_id=None, task_id=None) -> dict:
    message = {
        "messageId": demo_uuid(),
        "role": "ROLE_USER",
        "parts": [{"text": text, "mediaType": "text/plain"}],
    }
    if context_id is not None:
        message["contextId"] = context_id
    if task_id is not None:
        message["taskId"] = task_id
    return {
        "jsonrpc": "2.0",
        "id": demo_uuid(),
        "method": "SendMessage",
        "params": {"message": message},
    }

def a2a_teaching_check_request(request: dict) -> None:
    assert request.get("jsonrpc") == "2.0" and request.get("method") == "SendMessage"
    message = request["params"]["message"]
    assert message["role"] == "ROLE_USER" and message["messageId"]
    assert message["parts"] and all(sum(k in p for k in ("text", "raw", "url", "data")) == 1
                                    for p in message["parts"])

# In-memory state belongs to the simulated REMOTE agent. It never crosses the wire.
_demo_remote_tasks = {}

def demo_market_agent(request: dict) -> dict:
    """Simulated remote A2A server. Its prompts/models/graph remain opaque."""
    a2a_teaching_check_request(request)
    message = request["params"]["message"]
    text = " ".join(p["text"] for p in message["parts"] if "text" in p).lower()
    context_id = message.get("contextId") or demo_uuid()
    task_id = message.get("taskId") or demo_uuid()

    if message.get("taskId") and task_id not in _demo_remote_tasks:
        return {"jsonrpc": "2.0", "id": request["id"], "error": {
            "code": -32001, "message": "Task not found"
        }}

    if "report" not in text:
        task = {
            "id": task_id,
            "contextId": context_id,
            "status": {
                "state": "TASK_STATE_REJECTED",
                "message": {
                    "messageId": demo_uuid(),
                    "contextId": context_id,
                    "taskId": task_id,
                    "role": "ROLE_AGENT",
                    "parts": [{"text": "This request is outside my advertised report-quote skill."}],
                },
            },
        }
    else:
        quote = {
            "sku": DEMO_ONLY_REPORT_SKU,
            "title": "Q3 2026 EU semiconductor demand report",
            "amountMinor": DEMO_ONLY_REPORT_PRICE_MINOR,
            "currency": "USD",
            "merchant": DEMO_ONLY_MARKET_MERCHANT,
        }
        task = {
            "id": task_id,
            "contextId": context_id,
            "status": {"state": "TASK_STATE_COMPLETED"},
            "artifacts": [{
                "artifactId": demo_uuid(),
                "name": "Market-report quote",
                "parts": [
                    {"text": "Quote prepared for the requested report.", "mediaType": "text/plain"},
                    {"data": quote, "mediaType": "application/json"},
                ],
            }],
        }

    _demo_remote_tasks[task_id] = task
    return {"jsonrpc": "2.0", "id": request["id"], "result": {"task": task}}

def a2a_send(text: str, *, context_id=None, task_id=None) -> dict:
    return demo_market_agent(a2a_build_send_message(
        text, context_id=context_id, task_id=task_id
    ))

a2a_teaching_check_card(DEMO_ONLY_MARKET_AGENT_CARD)
preferred = DEMO_ONLY_MARKET_AGENT_CARD["supportedInterfaces"][0]
print("PASS - current Agent Card fields are present.")
print("preferred interface:", preferred)
print("authentication: none in this offline fixture; nothing to obtain")

"""
EXPECTED OUTPUT
---------------
PASS - current Agent Card fields are present.
preferred interface: {'url': 'https://agents.marketdata.example/a2a/rpc',
                      'protocolBinding': 'JSONRPC', 'protocolVersion': '1.0'}
authentication: none in this offline fixture; nothing to obtain
"""


In [ ]:
# See A2A work from the CLIENT'S point of view.
a2a_first_request = a2a_build_send_message(
    "Quote the Q3 2026 EU semiconductor demand report."
)
a2a_first = demo_market_agent(a2a_first_request)
a2a_first_task = a2a_first["result"]["task"]
a2a_quote = a2a_first_task["artifacts"][0]["parts"][1]["data"]

print("REQUEST (first turn; client sends no contextId or taskId):")
print(json.dumps(a2a_first_request, indent=2)[:520])
print("\nRESPONSE PATH: result.task")
print("state:", a2a_first_task["status"]["state"])
print("quote:", a2a_quote)

assert a2a_first_request["method"] == "SendMessage"
assert "kind" not in json.dumps(a2a_first_request), "v1 Parts are flattened"
assert a2a_first_task["status"]["state"] == "TASK_STATE_COMPLETED"
assert a2a_first_task["artifacts"][0]["artifactId"]
assert a2a_quote["amountMinor"] == 24_000
print("PASS - v1 method, enums, flattened Parts, response wrapper and artifactId are present.")

# Same conversation, NEW task: carry contextId and omit taskId.
a2a_second = a2a_send(
    "Quote the same report with a Q2 comparison.",
    context_id=a2a_first_task["contextId"],
)
a2a_second_task = a2a_second["result"]["task"]
assert a2a_second_task["contextId"] == a2a_first_task["contextId"]
assert a2a_second_task["id"] != a2a_first_task["id"]
print("PASS - same contextId + no taskId created a new Task in the same conversation.")

# Unsupported work is not 'method not found': SendMessage exists, but the Task is rejected.
a2a_bad = a2a_send("Book me a flight to Munich.")
assert a2a_bad["result"]["task"]["status"]["state"] == "TASK_STATE_REJECTED"
print("PASS - out-of-scope work returned TASK_STATE_REJECTED, not JSON-RPC -32601.")

# Only protocol objects cross the boundary—not the vendor's implementation.
a2a_wire = json.dumps(a2a_first)
assert all(secret_word not in a2a_wire for secret_word in ("system_prompt", "model_name", "graph_state"))
print("PASS - the wire contains a Task and Artifact, not the vendor's internals.")

"""
EXPECTED OUTPUT
---------------
REQUEST ... "method": "SendMessage" ... "role": "ROLE_USER" ... "parts": [{"text": ...}]
RESPONSE PATH: result.task
state: TASK_STATE_COMPLETED
quote: {... 'amountMinor': 24000, 'currency': 'USD', ...}
PASS - v1 method, enums, flattened Parts, response wrapper and artifactId are present.
PASS - same contextId + no taskId created a new Task in the same conversation.
PASS - out-of-scope work returned TASK_STATE_REJECTED, not JSON-RPC -32601.
PASS - the wire contains a Task and Artifact, not the vendor's internals.
"""


## AP2 — prove the user authorised the spend

AP2 v0.2 secures **agent-performed payment transactions**. It operates as a security feature inside a commerce protocol; catalog search, checkout APIs and the transport between roles are outside AP2’s scope.

### Five roles

| Role | Responsibility here |
|---|---|
| **Shopping Agent (SA)** | Finds the report, assembles mandate content and acts within delegated constraints |
| **Trusted Surface (TS)** | Non-agentic UI that shows the authority to the user, authenticates them and obtains consent |
| **Merchant (M)** | Creates and signs the checkout; verifies the Checkout Mandate |
| **Credential Provider (CP)** | Verifies payment authority and supplies a payment credential |
| **Merchant Payment Processor (MPP)** | Processes the merchant’s payment and returns a Payment Receipt |

The Trusted Surface **must be non-agentic**, and AP2 verification must run in deterministic code even when a role uses an LLM for communication.

### Two mandate types, each open or closed

| Evidence | Open form (before the exact purchase exists) | Closed form (one exact purchase) |
|---|---|---|
| **Checkout Mandate** | `vct = mandate.checkout.open.1`; constrains allowed merchants and line items | `vct = mandate.checkout.1`; contains `checkout_jwt` and `checkout_hash` |
| **Payment Mandate** | `vct = mandate.payment.open.1`; constrains payee, amount, instrument, execution and its referenced open Checkout Mandate | `vct = mandate.payment.1`; contains `transaction_id`, payee, integer-minor-unit amount and payment instrument |

- **Direct / human present:** the user sees and signs the closed Checkout and Payment Mandates.
- **Autonomous / human not present:** the user first signs open mandates on a Trusted Surface. Each includes the agent public key in `cnf`. Later, that agent signs closed mandates for a purchase that deterministic verifiers prove satisfies the open constraints.

This appendix uses the autonomous flow. The user is present **once**, when granting a 30-minute authority. There is no fake “auto-approval” later. If a constraint cannot be resolved, the flow declines and may bring the user back for a direct approval.

AP2 issues no universal “AP2 key.” In a real deployment, user credentials and agent keys are enrolled under a trust model; merchant and processor keys are resolved through trusted infrastructure. The next cell generates temporary identities so the evidence chain can be tested offline.

Official references: [AP2 v0.2 specification](https://ap2-protocol.org/ap2/specification/) · [flows](https://ap2-protocol.org/ap2/flows/) · [Checkout Mandate](https://ap2-protocol.org/ap2/checkout_mandate/) · [Payment Mandate](https://ap2-protocol.org/ap2/payment_mandate/) · [releases](https://github.com/google-agentic-commerce/AP2/releases)


In [ ]:
# Local cryptographic teaching wrapper.
# It uses real ECDSA P-256 signatures, but simple JWS—not AP2's full SD-JWT/KB-SD-JWT chain.
import base64
import hashlib
import time
from copy import deepcopy
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.utils import (
    decode_dss_signature, encode_dss_signature,
)

def demo_b64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode("ascii")

def demo_b64url_decode(value: str) -> bytes:
    return base64.urlsafe_b64decode(value + "=" * (-len(value) % 4))

def demo_json_bytes(value) -> bytes:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()

def demo_public_jwk(private_key) -> dict:
    numbers = private_key.public_key().public_numbers()
    return {
        "kty": "EC", "crv": "P-256",
        "x": demo_b64url(numbers.x.to_bytes(32, "big")),
        "y": demo_b64url(numbers.y.to_bytes(32, "big")),
    }

def demo_jwk_to_public_key(jwk: dict):
    assert jwk["kty"] == "EC" and jwk["crv"] == "P-256"
    x = int.from_bytes(demo_b64url_decode(jwk["x"]), "big")
    y = int.from_bytes(demo_b64url_decode(jwk["y"]), "big")
    return ec.EllipticCurvePublicNumbers(x, y, ec.SECP256R1()).public_key()

def demo_jws_sign(claims: dict, private_key, *, kid: str, typ: str = "JWT") -> str:
    header = {"alg": "ES256", "kid": kid, "typ": typ}
    encoded_header = demo_b64url(demo_json_bytes(header))
    encoded_claims = demo_b64url(demo_json_bytes(claims))
    signing_input = f"{encoded_header}.{encoded_claims}".encode("ascii")
    der = private_key.sign(signing_input, ec.ECDSA(hashes.SHA256()))
    r, s = decode_dss_signature(der)
    raw_signature = r.to_bytes(32, "big") + s.to_bytes(32, "big")
    return f"{encoded_header}.{encoded_claims}.{demo_b64url(raw_signature)}"

def demo_jws_unverified_claims(token: str) -> dict:
    pieces = token.split(".")
    if len(pieces) != 3:
        raise ValueError("not a compact JWS")
    return json.loads(demo_b64url_decode(pieces[1]))

def demo_jws_verify(token: str, public_jwk: dict) -> dict:
    encoded_header, encoded_claims, encoded_signature = token.split(".")
    raw_signature = demo_b64url_decode(encoded_signature)
    if len(raw_signature) != 64:
        raise ValueError("ES256 signature must be 64 bytes")
    r = int.from_bytes(raw_signature[:32], "big")
    s = int.from_bytes(raw_signature[32:], "big")
    der = encode_dss_signature(r, s)
    demo_jwk_to_public_key(public_jwk).verify(
        der,
        f"{encoded_header}.{encoded_claims}".encode("ascii"),
        ec.ECDSA(hashes.SHA256()),
    )
    return json.loads(demo_b64url_decode(encoded_claims))

def demo_token_hash(token: str) -> str:
    return demo_b64url(hashlib.sha256(token.encode("ascii")).digest())

# Fresh in-memory identities. No participant supplies these; rerunning the cell replaces them.
_demo_user_private_key = ec.generate_private_key(ec.SECP256R1())
_demo_agent_private_key = ec.generate_private_key(ec.SECP256R1())
_demo_market_merchant_private_key = ec.generate_private_key(ec.SECP256R1())
_demo_wrong_merchant_private_key = ec.generate_private_key(ec.SECP256R1())
_demo_mpp_private_key = ec.generate_private_key(ec.SECP256R1())

DEMO_ONLY_USER_PUBLIC_JWK = demo_public_jwk(_demo_user_private_key)
DEMO_ONLY_AGENT_PUBLIC_JWK = demo_public_jwk(_demo_agent_private_key)
DEMO_ONLY_MPP_PUBLIC_JWK = demo_public_jwk(_demo_mpp_private_key)
DEMO_ONLY_PAYMENT_INSTRUMENT = {
    "id": "instrument_demo_4242", "type": "card", "description": "Demo card •••• 4242"
}
DEMO_ONLY_WRONG_MERCHANT = {
    "id": "merchant_wrong_demo", "name": "Wrong Demo Merchant",
    "website": "https://wrong-merchant.example",
}
DEMO_ONLY_MERCHANT_TRUST = {
    DEMO_ONLY_MARKET_MERCHANT["id"]: demo_public_jwk(_demo_market_merchant_private_key),
    DEMO_ONLY_WRONG_MERCHANT["id"]: demo_public_jwk(_demo_wrong_merchant_private_key),
}

print("PASS - generated ephemeral P-256 identities in memory.")
print("participant-supplied keys: 0")
print("credential format: simplified local JWS wrapper (not AP2 SD-JWT)")

"""
EXPECTED OUTPUT
---------------
PASS - generated ephemeral P-256 identities in memory.
participant-supplied keys: 0
credential format: simplified local JWS wrapper (not AP2 SD-JWT)
"""


In [ ]:
# The Trusted Surface creates USER-SIGNED open Checkout and Payment Mandates.
# These are the constraints under which the user permits autonomous action.
def ap2_make_open_mandates(max_amount_minor: int, *, ttl_seconds: int = 30 * 60) -> dict:
    if type(max_amount_minor) is not int or max_amount_minor < 0:
        raise ValueError("max_amount_minor must be a non-negative integer")
    now = int(time.time())
    expires = now + ttl_seconds

    open_checkout_claims = {
        "vct": "mandate.checkout.open.1",
        "constraints": [
            {
                "type": "checkout.allowed_merchants",
                "allowed": [DEMO_ONLY_MARKET_MERCHANT],
            },
            {
                "type": "checkout.line_items",
                "items": [{
                    "id": "required-report",
                    "acceptable_items": [{
                        "id": DEMO_ONLY_REPORT_SKU,
                        "title": "Q3 2026 EU semiconductor demand report",
                    }],
                    "quantity": 1,
                }],
            },
        ],
        "cnf": {"jwk": DEMO_ONLY_AGENT_PUBLIC_JWK},
        "iat": now,
        "exp": expires,
    }
    open_checkout_token = demo_jws_sign(
        open_checkout_claims, _demo_user_private_key,
        kid="demo-user", typ="demo+open-checkout+jws",
    )

    open_payment_claims = {
        "vct": "mandate.payment.open.1",
        "constraints": [
            {
                "type": "payment.amount_range",
                "currency": "USD", "min": 0, "max": max_amount_minor,
            },
            {
                "type": "payment.allowed_payees",
                "allowed": [DEMO_ONLY_MARKET_MERCHANT],
            },
            {
                "type": "payment.allowed_payment_instruments",
                "allowed": [DEMO_ONLY_PAYMENT_INSTRUMENT],
            },
            {
                "type": "payment.reference",
                # In full AP2 this is the digest of the associated open Checkout Mandate.
                "conditional_transaction_id": demo_token_hash(open_checkout_token),
            },
        ],
        "cnf": {"jwk": DEMO_ONLY_AGENT_PUBLIC_JWK},
        "iat": now,
        "exp": expires,
    }
    open_payment_token = demo_jws_sign(
        open_payment_claims, _demo_user_private_key,
        kid="demo-user", typ="demo+open-payment+jws",
    )
    return {
        "checkout_token": open_checkout_token,
        "payment_token": open_payment_token,
    }

ap2_demo_open = ap2_make_open_mandates(50_000)  # USD 500.00
demo_open_checkout_claims = demo_jws_verify(
    ap2_demo_open["checkout_token"], DEMO_ONLY_USER_PUBLIC_JWK
)
demo_open_payment_claims = demo_jws_verify(
    ap2_demo_open["payment_token"], DEMO_ONLY_USER_PUBLIC_JWK
)
amount_rule = next(c for c in demo_open_payment_claims["constraints"]
                   if c["type"] == "payment.amount_range")

assert demo_open_checkout_claims["vct"] == "mandate.checkout.open.1"
assert demo_open_payment_claims["vct"] == "mandate.payment.open.1"
assert demo_open_checkout_claims["cnf"]["jwk"] == DEMO_ONLY_AGENT_PUBLIC_JWK
assert amount_rule["max"] == 50_000 and amount_rule["currency"] == "USD"
print("PASS - user-signed open mandates verified.")
print("authority: one report, one merchant, one instrument, USD 500.00, 30 minutes")
print("agent binding (cnf):", demo_open_checkout_claims["cnf"]["jwk"]["crv"])

"""
EXPECTED OUTPUT
---------------
PASS - user-signed open mandates verified.
authority: one report, one merchant, one instrument, USD 500.00, 30 minutes
agent binding (cnf): P-256
"""


In [ ]:
# The A2A quote becomes the merchant Checkout. The quote is not discarded or rebuilt from globals.
# Checkout contents belong to the commerce protocol; AP2 requires the Merchant to sign the Checkout JWT.
def demo_make_merchant_checkout_jwt(quote: dict, *, merchant=None, private_key=None) -> str:
    merchant = merchant or quote["merchant"]
    private_key = private_key or _demo_market_merchant_private_key
    amount_minor = quote["amountMinor"]
    if type(amount_minor) is not int or amount_minor <= 0:
        raise ValueError("quote amount must be a positive integer in minor units")
    checkout = {
        "checkout_id": demo_uuid(),
        "merchant": merchant,
        "line_items": [{
            "id": quote["sku"], "title": quote["title"],
            "unit_amount_minor": amount_minor, "quantity": 1,
        }],
        "total_amount_minor": amount_minor,
        "currency": quote["currency"],
        "fulfillment": "digital-download",
    }
    return demo_jws_sign(
        checkout, private_key, kid=f"{merchant['id']}-checkout", typ="JWT"
    )

def ap2_make_closed_mandates(open_mandates: dict, checkout_jwt: str,
                              *, payment_instrument=None) -> dict:
    payment_instrument = payment_instrument or DEMO_ONLY_PAYMENT_INSTRUMENT
    checkout_unverified = demo_jws_unverified_claims(checkout_jwt)
    merchant_id = checkout_unverified["merchant"]["id"]
    merchant_jwk = DEMO_ONLY_MERCHANT_TRUST[merchant_id]
    checkout = demo_jws_verify(checkout_jwt, merchant_jwk)
    checkout_hash = demo_token_hash(checkout_jwt)
    now = int(time.time())

    closed_checkout_claims = {
        "vct": "mandate.checkout.1",
        "checkout_jwt": checkout_jwt,
        "checkout_hash": checkout_hash,
        "iat": now,
        "exp": now + 10 * 60,
    }
    closed_payment_claims = {
        "vct": "mandate.payment.1",
        "transaction_id": checkout_hash,
        "payee": checkout["merchant"],
        "payment_amount": {
            "amount": checkout["total_amount_minor"],
            "currency": checkout["currency"],
        },
        "payment_instrument": payment_instrument,
        "iat": now,
        "exp": now + 10 * 60,
    }
    closed_checkout_token = demo_jws_sign(
        closed_checkout_claims, _demo_agent_private_key,
        kid="demo-shopping-agent", typ="demo+closed-checkout+jws",
    )
    closed_payment_token = demo_jws_sign(
        closed_payment_claims, _demo_agent_private_key,
        kid="demo-shopping-agent", typ="demo+closed-payment+jws",
    )
    # Notebook-only stand-in for KB-SD-JWT's cryptographic open→closed binding.
    # It is deliberately outside the official mandate claim sets and named DEMO.
    demo_binding_claims = {
        "demo_type": "notebook.open-closed-binding.1",
        "open_checkout_hash": demo_token_hash(open_mandates["checkout_token"]),
        "open_payment_hash": demo_token_hash(open_mandates["payment_token"]),
        "closed_checkout_hash": demo_token_hash(closed_checkout_token),
        "closed_payment_hash": demo_token_hash(closed_payment_token),
    }
    return {
        "checkout_token": closed_checkout_token,
        "payment_token": closed_payment_token,
        "demo_binding_token": demo_jws_sign(
            demo_binding_claims, _demo_agent_private_key,
            kid="demo-shopping-agent", typ="demo+open-closed-binding+jws",
        ),
    }

ap2_demo_checkout_jwt = demo_make_merchant_checkout_jwt(a2a_quote)
ap2_demo_closed = ap2_make_closed_mandates(ap2_demo_open, ap2_demo_checkout_jwt)
demo_closed_checkout = demo_jws_verify(
    ap2_demo_closed["checkout_token"], DEMO_ONLY_AGENT_PUBLIC_JWK
)
demo_closed_payment = demo_jws_verify(
    ap2_demo_closed["payment_token"], DEMO_ONLY_AGENT_PUBLIC_JWK
)
demo_checkout = demo_jws_verify(
    ap2_demo_checkout_jwt,
    DEMO_ONLY_MERCHANT_TRUST[DEMO_ONLY_MARKET_MERCHANT["id"]],
)

assert demo_checkout["line_items"][0]["id"] == a2a_quote["sku"]
assert demo_checkout["total_amount_minor"] == a2a_quote["amountMinor"]
assert demo_closed_checkout["checkout_hash"] == demo_token_hash(ap2_demo_checkout_jwt)
assert demo_closed_payment["transaction_id"] == demo_closed_checkout["checkout_hash"]
demo_binding = demo_jws_verify(
    ap2_demo_closed["demo_binding_token"], DEMO_ONLY_AGENT_PUBLIC_JWK
)
assert demo_binding["open_checkout_hash"] == demo_token_hash(ap2_demo_open["checkout_token"])
assert demo_binding["closed_payment_hash"] == demo_token_hash(ap2_demo_closed["payment_token"])
print("PASS - the exact A2A quote became a merchant-signed Checkout JWT.")
print("PASS - agent-signed closed Checkout and Payment Mandates share the checkout hash.")
print("PASS - notebook-only binding token links the open and closed evidence pairs.")
print("closed vct values:", demo_closed_checkout["vct"], "/", demo_closed_payment["vct"])

"""
EXPECTED OUTPUT
---------------
PASS - the exact A2A quote became a merchant-signed Checkout JWT.
PASS - agent-signed closed Checkout and Payment Mandates share the checkout hash.
PASS - notebook-only binding token links the open and closed evidence pairs.
closed vct values: mandate.checkout.1 / mandate.payment.1
"""


In [ ]:
# Deterministic AP2 verifier and signed receipts.
# The model is deliberately absent: authorization is a set of invariants, not an opinion.
def ap2_constraint(claims: dict, constraint_type: str):
    return next((c for c in claims.get("constraints", [])
                 if c.get("type") == constraint_type), None)

def ap2_chain_id(open_mandates: dict) -> str:
    joined = open_mandates["checkout_token"] + "." + open_mandates["payment_token"]
    return demo_b64url(hashlib.sha256(joined.encode()).digest())

def ap2_verify_chain(open_mandates: dict, closed_mandates: dict, *, now=None,
                     consumed_chain_ids=None) -> list[str]:
    now = int(time.time()) if now is None else now
    failures = []

    def verify(label, token, jwk):
        try:
            return demo_jws_verify(token, jwk)
        except (InvalidSignature, ValueError, KeyError, AssertionError, json.JSONDecodeError):
            failures.append(f"{label} signature or structure invalid")
            return None

    open_checkout = verify("open Checkout Mandate", open_mandates["checkout_token"],
                           DEMO_ONLY_USER_PUBLIC_JWK)
    open_payment = verify("open Payment Mandate", open_mandates["payment_token"],
                          DEMO_ONLY_USER_PUBLIC_JWK)
    if not open_checkout or not open_payment:
        return failures

    if open_checkout.get("vct") != "mandate.checkout.open.1":
        failures.append("open Checkout Mandate vct mismatch")
    if open_payment.get("vct") != "mandate.payment.open.1":
        failures.append("open Payment Mandate vct mismatch")
    if now >= open_checkout.get("exp", 0) or now >= open_payment.get("exp", 0):
        failures.append("open mandate authority expired")

    checkout_cnf = open_checkout.get("cnf", {}).get("jwk")
    payment_cnf = open_payment.get("cnf", {}).get("jwk")
    if checkout_cnf != payment_cnf or checkout_cnf != DEMO_ONLY_AGENT_PUBLIC_JWK:
        failures.append("open mandates do not bind the authorized shopping-agent key")
        return failures

    closed_checkout = verify("closed Checkout Mandate", closed_mandates["checkout_token"],
                             checkout_cnf)
    closed_payment = verify("closed Payment Mandate", closed_mandates["payment_token"],
                            checkout_cnf)
    if not closed_checkout or not closed_payment:
        return failures

    demo_binding = verify("notebook open-to-closed binding",
                          closed_mandates.get("demo_binding_token", ""), checkout_cnf)
    if not demo_binding:
        return failures
    expected_demo_binding = {
        "demo_type": "notebook.open-closed-binding.1",
        "open_checkout_hash": demo_token_hash(open_mandates["checkout_token"]),
        "open_payment_hash": demo_token_hash(open_mandates["payment_token"]),
        "closed_checkout_hash": demo_token_hash(closed_mandates["checkout_token"]),
        "closed_payment_hash": demo_token_hash(closed_mandates["payment_token"]),
    }
    if demo_binding != expected_demo_binding:
        failures.append("notebook open-to-closed binding mismatch")

    if closed_checkout.get("vct") != "mandate.checkout.1":
        failures.append("closed Checkout Mandate vct mismatch")
    if closed_payment.get("vct") != "mandate.payment.1":
        failures.append("closed Payment Mandate vct mismatch")
    if now >= closed_checkout.get("exp", 0) or now >= closed_payment.get("exp", 0):
        failures.append("closed mandate expired")

    checkout_jwt = closed_checkout.get("checkout_jwt", "")
    try:
        checkout_unverified = demo_jws_unverified_claims(checkout_jwt)
        merchant_id = checkout_unverified["merchant"]["id"]
        merchant_jwk = DEMO_ONLY_MERCHANT_TRUST[merchant_id]
        checkout = demo_jws_verify(checkout_jwt, merchant_jwk)
    except (InvalidSignature, ValueError, KeyError, AssertionError, json.JSONDecodeError):
        failures.append("merchant Checkout JWT signature or trust binding invalid")
        return failures

    checkout_hash = demo_token_hash(checkout_jwt)
    if closed_checkout.get("checkout_hash") != checkout_hash:
        failures.append("closed Checkout Mandate checkout_hash mismatch")
    if closed_payment.get("transaction_id") != checkout_hash:
        failures.append("closed Payment Mandate transaction_id does not bind this checkout")

    line_items = checkout.get("line_items", [])
    valid_numbers = bool(line_items) and all(
        type(item.get("unit_amount_minor")) is int
        and type(item.get("quantity")) is int
        and item["unit_amount_minor"] > 0 and item["quantity"] > 0
        for item in line_items
    )
    if not valid_numbers:
        failures.append("checkout money and quantities must be positive integers")
    else:
        recomputed_total = sum(i["unit_amount_minor"] * i["quantity"] for i in line_items)
        if recomputed_total != checkout.get("total_amount_minor"):
            failures.append("checkout total does not equal its line items")

    allowed_merchants = ap2_constraint(open_checkout, "checkout.allowed_merchants")
    allowed_ids = {m["id"] for m in (allowed_merchants or {}).get("allowed", [])}
    if checkout["merchant"]["id"] not in allowed_ids:
        failures.append("checkout merchant is not allowed")

    required_items = ap2_constraint(open_checkout, "checkout.line_items")
    requirements = (required_items or {}).get("items", [])
    expected = {(choice["id"], requirement["quantity"])
                for requirement in requirements
                for choice in requirement.get("acceptable_items", [])}
    actual = {(item["id"], item["quantity"]) for item in line_items}
    if actual != expected:
        failures.append("checkout line items do not match the authorized report and quantity")

    payment_amount = closed_payment.get("payment_amount", {})
    if type(payment_amount.get("amount")) is not int or payment_amount.get("amount", -1) < 0:
        failures.append("payment amount must be a non-negative integer in minor units")
    if (payment_amount.get("amount") != checkout.get("total_amount_minor")
            or payment_amount.get("currency") != checkout.get("currency")):
        failures.append("payment amount or currency does not match the checkout")
    if closed_payment.get("payee") != checkout.get("merchant"):
        failures.append("payment payee does not match the checkout merchant")

    amount_range = ap2_constraint(open_payment, "payment.amount_range") or {}
    amount = payment_amount.get("amount")
    if (type(amount) is int and
            not (amount_range.get("min", 0) <= amount <= amount_range.get("max", -1))):
        failures.append("payment amount is outside the user-authorized range")
    if payment_amount.get("currency") != amount_range.get("currency"):
        failures.append("payment currency is not authorized")

    payee_rule = ap2_constraint(open_payment, "payment.allowed_payees") or {}
    if closed_payment.get("payee") not in payee_rule.get("allowed", []):
        failures.append("payment payee is not allowed")
    instrument_rule = ap2_constraint(open_payment, "payment.allowed_payment_instruments") or {}
    if closed_payment.get("payment_instrument") not in instrument_rule.get("allowed", []):
        failures.append("payment instrument is not allowed")
    reference_rule = ap2_constraint(open_payment, "payment.reference") or {}
    if reference_rule.get("conditional_transaction_id") != demo_token_hash(
            open_mandates["checkout_token"]):
        failures.append("open Payment Mandate does not reference the open Checkout Mandate")

    if consumed_chain_ids is not None and ap2_chain_id(open_mandates) in consumed_chain_ids:
        failures.append("open mandate pair has already been consumed")
    return failures

def demo_make_receipts(closed_mandates: dict, failures: list[str]) -> dict:
    now = int(time.time())
    status = "Error" if failures else "Success"
    checkout_receipt = {
        "status": status,
        "iss": DEMO_ONLY_MARKET_MERCHANT["id"],
        "iat": now,
        "reference": demo_token_hash(closed_mandates["checkout_token"]),
    }
    payment_receipt = {
        "status": status,
        "iss": "demo-mpp",
        "iat": now,
        "reference": demo_token_hash(closed_mandates["payment_token"]),
        "payment_id": demo_uuid(),
    }
    if failures:
        checkout_receipt.update(error="constraint_failure", error_description="; ".join(failures))
        payment_receipt.update(error="constraint_failure", error_description="; ".join(failures))
    else:
        checkout_receipt["order_id"] = demo_uuid()
        payment_receipt["psp_confirmation_id"] = demo_uuid()
    return {
        "checkout": demo_jws_sign(
            checkout_receipt, _demo_market_merchant_private_key,
            kid="demo-merchant-receipt", typ="demo+checkout-receipt+jws",
        ),
        "payment": demo_jws_sign(
            payment_receipt, _demo_mpp_private_key,
            kid="demo-mpp-receipt", typ="demo+payment-receipt+jws",
        ),
    }

def ap2_decide(open_mandates: dict, closed_mandates: dict, *, consumed_chain_ids=None) -> dict:
    failures = ap2_verify_chain(
        open_mandates, closed_mandates, consumed_chain_ids=consumed_chain_ids
    )
    receipts = demo_make_receipts(closed_mandates, failures)
    if not failures and consumed_chain_ids is not None:
        consumed_chain_ids.add(ap2_chain_id(open_mandates))
    return {
        "decision": "declined" if failures else "settled",
        "reasons": failures,
        "receipts": receipts,
    }

print("deterministic verifier ready: signatures, expiry, cnf, constraints, bindings, totals, replay")
print("signed receipts ready: Checkout Receipt + Payment Receipt, success or error")

"""
EXPECTED OUTPUT
---------------
deterministic verifier ready: signatures, expiry, cnf, constraints, bindings, totals, replay
signed receipts ready: Checkout Receipt + Payment Receipt, success or error
"""


In [ ]:
# Negative security tests. A verifier that only demonstrates success has taught very little.
def demo_tamper_jws_claims(token: str, mutate) -> str:
    """Change signed claims but keep the old signature: a deliberate attack fixture."""
    encoded_header, encoded_claims, encoded_signature = token.split(".")
    claims = json.loads(demo_b64url_decode(encoded_claims))
    mutate(claims)
    return f"{encoded_header}.{demo_b64url(demo_json_bytes(claims))}.{encoded_signature}"

def demo_agent_replace_closed_checkout_jwt(closed_mandates: dict, checkout_jwt: str) -> dict:
    """A compromised agent re-signs a closed mandate around a merchant token it altered."""
    changed = deepcopy(closed_mandates)
    claims = demo_jws_verify(changed["checkout_token"], DEMO_ONLY_AGENT_PUBLIC_JWK)
    claims["checkout_jwt"] = checkout_jwt
    claims["checkout_hash"] = demo_token_hash(checkout_jwt)
    changed["checkout_token"] = demo_jws_sign(
        claims, _demo_agent_private_key, kid="demo-shopping-agent",
        typ="demo+closed-checkout+jws",
    )
    return changed

# Honest chain and receipt bindings.
demo_ledger = set()
honest = ap2_decide(ap2_demo_open, ap2_demo_closed, consumed_chain_ids=demo_ledger)
assert honest["decision"] == "settled" and not honest["reasons"]
checkout_receipt = demo_jws_verify(
    honest["receipts"]["checkout"],
    DEMO_ONLY_MERCHANT_TRUST[DEMO_ONLY_MARKET_MERCHANT["id"]],
)
payment_receipt = demo_jws_verify(honest["receipts"]["payment"], DEMO_ONLY_MPP_PUBLIC_JWK)
assert checkout_receipt["reference"] == demo_token_hash(ap2_demo_closed["checkout_token"])
assert payment_receipt["reference"] == demo_token_hash(ap2_demo_closed["payment_token"])
print("PASS - honest chain settled and both signed receipts bind to their closed mandates.")

# 1. The shopping agent changes a merchant-signed total after checkout.
tampered_checkout_jwt = demo_tamper_jws_claims(
    ap2_demo_checkout_jwt,
    lambda claims: claims.update(total_amount_minor=240_000),
)
tampered_closed = demo_agent_replace_closed_checkout_jwt(ap2_demo_closed, tampered_checkout_jwt)
reasons = ap2_verify_chain(ap2_demo_open, tampered_closed)
assert any("merchant Checkout JWT signature" in reason for reason in reasons)
print("PASS - changing the merchant-signed checkout breaks its signature.")

# 2. A valid checkout from a different trusted merchant is still outside the user's allow-list.
wrong_quote = deepcopy(a2a_quote)
wrong_quote["merchant"] = DEMO_ONLY_WRONG_MERCHANT
wrong_checkout_jwt = demo_make_merchant_checkout_jwt(
    wrong_quote, merchant=DEMO_ONLY_WRONG_MERCHANT,
    private_key=_demo_wrong_merchant_private_key,
)
wrong_closed = ap2_make_closed_mandates(ap2_demo_open, wrong_checkout_jwt)
reasons = ap2_verify_chain(ap2_demo_open, wrong_closed)
assert any("merchant is not allowed" in reason for reason in reasons)
print("PASS - a valid checkout from an unapproved merchant is rejected.")

# 3. Expiry ends authority; it is not the same thing as active revocation.
expired_open = ap2_make_open_mandates(50_000, ttl_seconds=-1)
expired_closed = ap2_make_closed_mandates(expired_open, ap2_demo_checkout_jwt)
assert "open mandate authority expired" in ap2_verify_chain(expired_open, expired_closed)
print("PASS - expired open authority is rejected.")

# 4. A closed Payment Mandate cannot be pointed at a different checkout.
swapped = deepcopy(ap2_demo_closed)
payment_claims = demo_jws_verify(swapped["payment_token"], DEMO_ONLY_AGENT_PUBLIC_JWK)
payment_claims["transaction_id"] = "different-checkout-hash"
swapped["payment_token"] = demo_jws_sign(
    payment_claims, _demo_agent_private_key, kid="demo-shopping-agent",
    typ="demo+closed-payment+jws",
)
reasons = ap2_verify_chain(ap2_demo_open, swapped)
assert any("transaction_id" in reason for reason in reasons)
print("PASS - checkout/payment binding mismatch is rejected.")

# 5. The cap is integer minor-unit data enforced by code, not prose interpreted by a model.
low_cap_open = ap2_make_open_mandates(15_000)  # USD 150.00
low_cap_closed = ap2_make_closed_mandates(low_cap_open, ap2_demo_checkout_jwt)
reasons = ap2_verify_chain(low_cap_open, low_cap_closed)
assert "payment amount is outside the user-authorized range" in reasons
print("PASS - USD 240.00 is rejected against a USD 150.00 cap.")

# 6. A successful open mandate pair is single-use in this teaching ledger.
replay = ap2_decide(ap2_demo_open, ap2_demo_closed, consumed_chain_ids=demo_ledger)
assert replay["decision"] == "declined"
assert "open mandate pair has already been consumed" in replay["reasons"]
replay_receipt = demo_jws_verify(replay["receipts"]["payment"], DEMO_ONLY_MPP_PUBLIC_JWK)
assert replay_receipt["status"] == "Error"
print("PASS - replay is rejected and returns a signed Error receipt.")

"""
EXPECTED OUTPUT
---------------
PASS - honest chain settled and both signed receipts bind to their closed mandates.
PASS - changing the merchant-signed checkout breaks its signature.
PASS - a valid checkout from an unapproved merchant is rejected.
PASS - expired open authority is rejected.
PASS - checkout/payment binding mismatch is rejected.
PASS - USD 240.00 is rejected against a USD 150.00 cap.
PASS - replay is rejected and returns a signed Error receipt.
"""


## See the two protocols working together

The integration boundary is intentionally narrow: **only the structured A2A quote moves into the commerce/AP2 flow**. No AP2-specific A2A header or invented DataPart key is required by AP2 v0.2.

```mermaid
flowchart TD
    A["Discover Agent Card"] --> B["A2A SendMessage"]
    B --> C["Structured quote"]
    C --> D["User-signed open mandates"]
    D --> E["Merchant-signed checkout"]
    E --> F["Agent-signed closed mandates"]
    F --> G{"Deterministic verifier"}
    G -->|valid| H["Settle + signed receipts"]
    G -->|invalid| I["Decline + error receipts"]
```

The model may help find a report or explain a choice. It may not raise the cap, redefine the merchant allow-list, waive an expired mandate, alter a signed checkout or decide that a replay is “probably fine.” Those are code-level invariants.


In [ ]:
# End-to-end orchestration: same vendor quote, two user-set caps, two deterministic outcomes.
def ap2_run_report_purchase(budget_minor: int) -> dict:
    log = []
    a2a_teaching_check_card(DEMO_ONLY_MARKET_AGENT_CARD)
    log.append("A2A discover: selected JSONRPC 1.0 interface")

    response = a2a_send("Quote the Q3 2026 EU semiconductor demand report.")
    task = response["result"]["task"]
    quote = task["artifacts"][0]["parts"][1]["data"]
    log.append(f"A2A quote: {quote['amountMinor']} minor {quote['currency']} from {quote['merchant']['id']}")

    open_mandates = ap2_make_open_mandates(budget_minor)
    log.append(f"AP2 open: user-authorized cap {budget_minor} minor USD on a Trusted Surface")

    # Critical data-flow assertion: the Checkout is derived from this exact quote object.
    checkout_jwt = demo_make_merchant_checkout_jwt(quote)
    checkout = demo_jws_verify(
        checkout_jwt, DEMO_ONLY_MERCHANT_TRUST[quote["merchant"]["id"]]
    )
    assert checkout["line_items"][0]["id"] == quote["sku"]
    assert checkout["total_amount_minor"] == quote["amountMinor"]
    log.append("Commerce: merchant signed a checkout derived from the exact A2A quote")

    closed_mandates = ap2_make_closed_mandates(open_mandates, checkout_jwt)
    log.append("AP2 closed: authorized agent signed Checkout and Payment Mandates")

    ledger = set()
    outcome = ap2_decide(open_mandates, closed_mandates, consumed_chain_ids=ledger)
    log.append("AP2 gate: " + ("all invariants passed" if not outcome["reasons"]
                               else "; ".join(outcome["reasons"])))
    log.append("Receipts: signed Checkout and Payment receipts returned")
    return {
        "quote": quote,
        "open": open_mandates,
        "checkout_jwt": checkout_jwt,
        "closed": closed_mandates,
        "outcome": outcome,
        "log": log,
    }

ap2_settled = ap2_run_report_purchase(50_000)  # USD 500.00
ap2_declined = ap2_run_report_purchase(15_000) # USD 150.00

for label, run in [("CAP USD 500.00", ap2_settled), ("CAP USD 150.00", ap2_declined)]:
    print(f"\n=== {label} -> {run['outcome']['decision'].upper()} ===")
    for line in run["log"]:
        print(" ", line)
    closed_checkout = demo_jws_verify(run["closed"]["checkout_token"], DEMO_ONLY_AGENT_PUBLIC_JWK)
    closed_payment = demo_jws_verify(run["closed"]["payment_token"], DEMO_ONLY_AGENT_PUBLIC_JWK)
    checkout_receipt = demo_jws_verify(
        run["outcome"]["receipts"]["checkout"],
        DEMO_ONLY_MERCHANT_TRUST[DEMO_ONLY_MARKET_MERCHANT["id"]],
    )
    payment_receipt = demo_jws_verify(
        run["outcome"]["receipts"]["payment"], DEMO_ONLY_MPP_PUBLIC_JWK
    )
    print("  AUDIT open checkout :", demo_token_hash(run["open"]["checkout_token"])[:14])
    print("        checkout JWT  :", closed_checkout["checkout_hash"][:14])
    print("        payment link  :", closed_payment["transaction_id"][:14])
    print("        receipt refs  :", checkout_receipt["reference"][:14],
          payment_receipt["reference"][:14], payment_receipt["status"])

assert ap2_settled["outcome"]["decision"] == "settled"
assert ap2_declined["outcome"]["decision"] == "declined"
assert ap2_settled["quote"] == ap2_declined["quote"]
assert "payment amount is outside the user-authorized range" in ap2_declined["outcome"]["reasons"]
print("\nPASS - same quote, two user-set caps, two outcomes, one deterministic verifier.")
print("PASS - A2A quote data—not a second set of global price constants—built each checkout.")

"""
EXPECTED OUTPUT
---------------
=== CAP USD 500.00 -> SETTLED ===
  ... AP2 gate: all invariants passed
  ... receipt refs ... Success

=== CAP USD 150.00 -> DECLINED ===
  ... AP2 gate: payment amount is outside the user-authorized range
  ... receipt refs ... Error

PASS - same quote, two user-set caps, two outcomes, one deterministic verifier.
PASS - A2A quote data—not a second set of global price constants—built each checkout.
"""


### What must change before going live

This appendix is a protocol-shaped security exercise, not a production starter. A real deployment needs the following architecture and trust setup.

| Area | Offline fixture here | Production responsibility |
|---|---|---|
| A2A discovery | Local Agent Card dictionary | Fetch and cache `/.well-known/agent-card.json`; verify version, interface, capability and—when relied upon—card signature |
| A2A transport | In-process function call | Official SDK/client over the selected JSON-RPC, HTTP+JSON or gRPC interface; TLS, timeouts, retries, idempotency and observability |
| A2A authentication | None declared | Obtain the operator-specific API key, bearer token, OAuth/OIDC grant or mTLS identity described by the Agent Card |
| User authority | Ephemeral notebook key | User credential or trusted-agent-provider enrollment; a non-agentic Trusted Surface that authenticates the user and records informed consent |
| Agent identity | Ephemeral notebook key + `cnf` | Protected agent key lifecycle, provider trust and rotation; bind the authorized public key in open mandates |
| Merchant trust | Local dictionary | Trusted merchant-key discovery/registry and verification of the signed Checkout JWT |
| Mandate credentials | Simplified compact JWS | AP2 v0.2 SD-JWT/KB-SD-JWT processing, selective disclosure, nonce/audience handling and open→closed credential chaining |
| Payments | No payment rail | Credential Provider, scoped payment credential, network/payment-processor integration and compliance controls |
| Evidence | Local signed receipts | Checkout and Payment Receipts, durable audit retention, retrieval for disputes and privacy-minimized disclosures |
| Replay/state | In-memory `set` | Atomic single-use/recurrence accounting, rejection receipts, concurrency protection, revocation and expiry handling |

### Where real values come from

- **A2A endpoint and auth:** the remote agent operator publishes the endpoint and schemes in its Agent Card; the operator’s onboarding or OAuth flow supplies the credential.
- **User credential:** issued/enrolled under the chosen AP2 trust model and held on a Trusted Surface.
- **Agent key:** generated and protected by the agent provider, then bound in the user-signed open mandates through `cnf`.
- **Merchant verification key:** obtained through the merchant/payment ecosystem’s trusted key-discovery process.
- **Payment instrument/credential:** sourced and scoped by the Credential Provider; it is not a string invented by the model.

### Final warnings

1. A schema-shaped object is not a protocol implementation. Use the official specifications, SDKs and conformance tooling.
2. Never place spend authority only in a prompt. Models may propose; deterministic verifiers authorize.
3. Expiry limits the lifetime of authority; it is not active revocation. Production needs both where the risk model requires them.
4. Do not reuse an open autonomous mandate for overlapping purchases without the receipt/state rules required by AP2.
5. Preserve only the disclosures required for verification; mandate evidence is sensitive.

References: [A2A v1.0.1 specification](https://a2a-protocol.org/latest/specification/) · [A2A releases](https://github.com/a2aproject/A2A/releases) · [A2A Inspector](https://github.com/a2aproject/a2a-inspector) · [AP2 v0.2 specification](https://ap2-protocol.org/ap2/specification/) · [AP2 reference project](https://github.com/google-agentic-commerce/AP2)


---
### What you should be able to do now

- Choose between supervisor, choreography and actor–critic, and state what each costs.
- Express an agent boundary as an enforced write permission and a scoped context slice, not a sentence in a prompt.
- Put a model in a control path *safely*: compute the legal moves in code, let the model choose among them, and validate before you act.
- Decide per node whether the answer is a **judgement** (use a model) or a **fact** (use code) — and know why a fact-checker built the second way is worth more than one built the first.
- Cap every loop-back edge, and route budget exhaustion to a human instead of a recursion limit.
- Build a FastMCP server whose docstrings and type hints form the API contract, sandbox it properly, and consume it from LangChain, from an agent, and from a LangGraph team.
- Test an LLM system by asserting invariants rather than transcripts.
- Place MCP, A2A and AP2 on one map: agent→tool, agent→agent, and cryptographic payment authority inside a commerce flow—and know that this scenario composes A2A with AP2 rather than making one depend on the other.
- Delegate a request to an agent you didn't build over A2A v1, using `supportedInterfaces`, `SendMessage`, and server-created `contextId`/`taskId` values with their correct lifecycle semantics.
- Gate an autonomous purchase on deterministic verification of user-signed open mandates, agent-signed closed mandates, checkout/payment bindings and signed receipts—never on a model's word for the budget.

### Pitfall table

| Symptom | Cause | Fix |
|---|---|---|
| `GraphRecursionError` in a supervisor graph | Star topology costs 2 supersteps per unit of work; default limit is 25 | Set `recursion_limit` deliberately; add a revision cap |
| Team loops between writer and critic forever | Loop-back edge with no budget | `revision_count` guard, and remove `writer` from the legal routes once it is spent |
| The graph never terminates after a revision | A control field with a reducer, so it can never be reset to `{}` | Reducers on *accumulating* fields only |
| An LLM router picks a step with nothing left to do | The model was asked an open question | Offer it `legal_routes(state)` and discard anything outside it |
| A critic "approves" a fabricated citation | The critic shares the producer's context | Scope the critic's read slice; deny it the brief |
| Approved document ships with unverified edits | A rewrite did not void prior approvals | Let the writer reset `fact_check` / `review` |
| An agent silently corrupts another's field | No write scoping | The `scoped()` decorator |
| Self-checks pass on Monday and fail on Tuesday | Asserting on generated prose | Assert invariants, not transcripts |
| MCP handshake fails on `initialize` | Client/server SDK version skew | Pin `mcp`/`fastmcp` in requirements for both sides |
| `io.UnsupportedOperation: fileno` when spawning a stdio server | Jupyter's `sys.stderr` is not a real file, and the child process needs a real descriptor | Pass a real file as `errlog` (see the client cell) |
| MCP client fails with a JSON parse error | A `print()` to stdout in a stdio server | Log to `stderr` only; `show_banner=False` |
| A2A follow-up lands in the wrong scope | Confused conversation continuity with Task continuity | Same `contextId`, no `taskId` starts a new Task in the conversation; same valid `taskId` continues that Task |
| An agent purchase settles above the user's cap | The cap was prompt text or the open Payment Mandate was not evaluated | Compare the closed Payment Mandate's integer-minor-unit amount with `payment.amount_range` in deterministic code |
| Server starts but the tool errors on file access | Relative path; the subprocess cwd differs from yours | Resolve paths from `__file__` |
| Server spawns but imports fail | Launched with a different interpreter | `command=sys.executable` |
| Tool result looks like `[{'type':'text', ...}]` | MCP carries content blocks, not Python objects | A `parse_tool_result()` helper |
| A path guard passes but the file is outside the root | Compared strings instead of resolved paths, or allowed a symlink | `resolve()` then `is_relative_to()`, and refuse symlinks |
| Tool is discovered but the model never calls it | Vague docstring / untyped parameters | The docstring IS the contract — write it for a stranger |
| Server state resets between calls | Each `get_tools()` call opened its own session | Hold one open with `async with client.session(...)` |

### Stretch goals (optional)

1. **Second server, one dict entry.** Write a tiny second FastMCP server (say, a metrics lookup) and add it to `MultiServerMCPClient` as one more key. Time it — that elapsed time *is* the M+N claim, measured.
2. **Choreography variant.** Remove the supervisor; let the fact-checker hand off to the writer directly with `Command(goto=...)`. Then try to reconstruct who decided what from the log. The difficulty is why production teams pick supervisors.
3. **Break the fact-checker on purpose.** Give it the brief in its context slice, then re-run the hallucinating-writer cell five times and count how often `[S9]` survives. That number is the value of read scoping, in your own data.
4. **Make the reviewer disagree with itself.** Raise `temperature` on the reviewer's call and re-run the team ten times. Count the escalations. That variance is what the deterministic `_structure_notes` floor was protecting you from.
5. **Human gate on the escalation path.** Wire Session 1's `interrupt()` into `escalate_node` with a durable checkpointer, so an escalated briefing waits for a named human instead of ending. That composition — supervisor team, MCP tools, durable human gate — is the full capstone shape.

### Capstone tie-in

This lab satisfies **Milestone 6 — "Specialised multi-agent team + MCP integration"**. The artefacts you can lift directly into your capstone: the `scoped()` permission decorator, the `supervisor_policy()` + `legal_routes()` pair, the revision budget with an escalation path, and `project_mcp_server.py` with its `_safe_path` sandbox. The appendix's `ap2_verify_chain()` gate is the same pattern applied to money: if your capstone lets an agent spend on a user's behalf, that is where to start.